# Prepare


In [ ]:
!pip -q install ortools pandas numpy tabulate

In [ ]:
import os
import math
import time
import random
from collections import defaultdict, Counter
from itertools import product

import numpy as np
import pandas as pd

from tabulate import tabulate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"

CNN_CSV   = os.path.join(BASE_DIR, "cnn_bench.csv")
VIT_CSV   = os.path.join(BASE_DIR, "vit_bench.csv")
BERT_CSV  = os.path.join(BASE_DIR, "bert_bench.csv")
GPT2_CSV  = os.path.join(BASE_DIR, "gpt2m_streaming_bench.csv")
LLAMA_CSV = os.path.join(BASE_DIR, "llama32_3b_streaming_bench.csv")

for p in [CNN_CSV, VIT_CSV, BERT_CSV, GPT2_CSV, LLAMA_CSV]:
    print(p, "->", os.path.exists(p))

/content/drive/MyDrive/Colab Notebooks/cnn_bench.csv -> True
/content/drive/MyDrive/Colab Notebooks/vit_bench.csv -> True
/content/drive/MyDrive/Colab Notebooks/bert_bench.csv -> True
/content/drive/MyDrive/Colab Notebooks/gpt2m_streaming_bench.csv -> True
/content/drive/MyDrive/Colab Notebooks/llama32_3b_streaming_bench.csv -> True


In [ ]:
PROFILE_ORDER = ["7g", "4g", "3g", "2g", "1g"]

PROFILE_MEM_MB = {
    "1g": 5 * 1024,
    "2g": 10 * 1024,
    "3g": 20 * 1024,
    "4g": 20 * 1024,
    "7g": 40 * 1024,
}

In [ ]:
TEMPLATES = [
    ("7",               (1,0,0,0,0)),
    ("4+3",             (0,1,1,0,0)),
    ("4+2+1",           (0,1,0,1,1)),
    ("4+1+1+1",         (0,1,0,0,3)),
    ("3+3+1",           (0,0,2,0,1)),
    ("3+2+1+1",         (0,0,1,1,2)),
    ("3+1+1+1+1",       (0,0,1,0,4)),
    ("2+2+3",           (0,0,1,2,0)),
    ("2+2+2+1",         (0,0,0,3,1)),
    ("2+2+1+1+1",       (0,0,0,2,3)),
    ("2+1+1+1+1+1",     (0,0,0,1,5)),
    ("1+1+1+1+1+1+1",   (0,0,0,0,7)),
]

TEMPLATE_K = []
for name, vec in TEMPLATES:
    TEMPLATE_K.append({
        "7g": vec[0],
        "4g": vec[1],
        "3g": vec[2],
        "2g": vec[3],
        "1g": vec[4],
    })

print("Number of templates =", len(TEMPLATES))
for i, (name, _) in enumerate(TEMPLATES):
    print(f"{i:2d}: {name:18s} -> {TEMPLATE_K[i]}")

Number of templates = 12
 0: 7                  -> {'7g': 1, '4g': 0, '3g': 0, '2g': 0, '1g': 0}
 1: 4+3                -> {'7g': 0, '4g': 1, '3g': 1, '2g': 0, '1g': 0}
 2: 4+2+1              -> {'7g': 0, '4g': 1, '3g': 0, '2g': 1, '1g': 1}
 3: 4+1+1+1            -> {'7g': 0, '4g': 1, '3g': 0, '2g': 0, '1g': 3}
 4: 3+3+1              -> {'7g': 0, '4g': 0, '3g': 2, '2g': 0, '1g': 1}
 5: 3+2+1+1            -> {'7g': 0, '4g': 0, '3g': 1, '2g': 1, '1g': 2}
 6: 3+1+1+1+1          -> {'7g': 0, '4g': 0, '3g': 1, '2g': 0, '1g': 4}
 7: 2+2+3              -> {'7g': 0, '4g': 0, '3g': 1, '2g': 2, '1g': 0}
 8: 2+2+2+1            -> {'7g': 0, '4g': 0, '3g': 0, '2g': 3, '1g': 1}
 9: 2+2+1+1+1          -> {'7g': 0, '4g': 0, '3g': 0, '2g': 2, '1g': 3}
10: 2+1+1+1+1+1        -> {'7g': 0, '4g': 0, '3g': 0, '2g': 1, '1g': 5}
11: 1+1+1+1+1+1+1      -> {'7g': 0, '4g': 0, '3g': 0, '2g': 0, '1g': 7}


In [ ]:
WORKLOAD_SPECS = [
    {
        "name": "llama",
        "family": "llm",
        "csv": LLAMA_CSV,
        "model_match": "Llama-3.2-3B-Instruct",
        "batch_candidates": None,   # auto-discover from CSV
        "prompt_len": 1024,
        "output_tokens": 64,
        "arrival_rate": 3,       # req/s
        "slo_type": "TTFT/TPOT",
        "ttft_slo_ms": 100.0,
        "tpot_slo_ms": 25.0,
    },
    {
        "name": "gpt2",
        "family": "llm",
        "csv": GPT2_CSV,
        "model_match": "gpt2-medium",
        "batch_candidates": None,
        "prompt_len": 64,
        "output_tokens": 64,
        "arrival_rate": 100, #20
        "slo_type": "TTFT/TPOT",
        "ttft_slo_ms": 20.0,
        "tpot_slo_ms": 15.0,
    },
    {
        "name": "vgg16",
        "family": "cv",
        "csv": CNN_CSV,
        "model_match": "vgg16",
        "batch_candidates": None,
        "arrival_rate": 3000.0, #300
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
    {
        "name": "resnet50",
        "family": "cv",
        "csv": CNN_CSV,
        "model_match": "resnet50",
        "batch_candidates": None,
        "arrival_rate": 3000.0, #200
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
    {
        "name": "vit_base",
        "family": "cv",
        "csv": VIT_CSV,
        "model_match": "vit-base-patch16-224",
        "batch_candidates": None,
        "arrival_rate": 3000.0, #350
        "slo_type": "E2E",
        "e2e_slo_ms": 50.0,
    },
]

WORKLOAD_NAMES = [w["name"] for w in WORKLOAD_SPECS]
N_WORKLOADS = len(WORKLOAD_SPECS)

In [ ]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_mig_name(s: str):
    """
    Map strings like:
      'NVIDIA A100-PCIE-40GB MIG 1g.5gb' -> '1g'
      'MIG 3g.20gb' -> '3g'
    """
    if not isinstance(s, str):
        return None
    s = s.lower().replace(" ", "")
    if "1g." in s or "1g" in s:
        return "1g"
    if "2g." in s or "2g" in s:
        return "2g"
    if "3g." in s or "3g" in s:
        return "3g"
    if "4g." in s or "4g" in s:
        return "4g"
    if "7g." in s or "7g" in s:
        return "7g"
    return None


def decode_tps_to_tpot_ms(decode_tps: float):
    """
    decode_tps = tokens / second
    TPOT(ms/token) = 1000 / decode_tps
    """
    if decode_tps is None or np.isnan(decode_tps) or decode_tps <= 0:
        return np.nan
    return 1000.0 / decode_tps


def safe_req_rate_from_latency_ms(lat_ms: float, batch: int):
    """
    service rate (req/s) = 1000 / latency_ms
    """
    if lat_ms is None or np.isnan(lat_ms) or lat_ms <= 0:
        return np.nan
    return batch * 1000.0 / lat_ms

In [ ]:
def discover_batch_candidates_for_spec(spec, profile_order):
    df = pd.read_csv(spec["csv"]).copy()

    mig_col = pick_col(df, ["mig_name"])
    model_col = pick_col(df, ["model"])
    batch_col = pick_col(df, ["batch"])

    if mig_col is None or model_col is None or batch_col is None:
        raise ValueError(f"Missing essential columns in {spec['csv']}")

    df["profile"] = df[mig_col].apply(normalize_mig_name)
    df = df[df["profile"].isin(profile_order)].copy()
    df = df[df[model_col] == spec["model_match"]].copy()

    if spec["family"] == "llm":
        prompt_col = pick_col(df, ["prompt_len"])
        output_col = pick_col(df, ["output_tokens"])
        if prompt_col is None or output_col is None:
            raise ValueError(f"Missing LLM columns in {spec['csv']}")
        df = df[df[prompt_col] == spec["prompt_len"]].copy()
        df = df[df[output_col] == spec["output_tokens"]].copy()

    batches = sorted([int(b) for b in df[batch_col].dropna().unique().tolist() if float(b) > 0])

    if spec.get("batch_candidates") is not None:
        allow = set(int(x) for x in spec["batch_candidates"])
        batches = [b for b in batches if b in allow]

    return batches

In [ ]:
BATCHES_PER_WORKLOAD = {}
for spec in WORKLOAD_SPECS:
    BATCHES_PER_WORKLOAD[spec["name"]] = discover_batch_candidates_for_spec(spec, PROFILE_ORDER)

print("=== Batch candidates per workload ===")
for k, v in BATCHES_PER_WORKLOAD.items():
    print(f"{k:10s} -> {v}")

BATCH_LEVELS = sorted(set(b for lst in BATCHES_PER_WORKLOAD.values() for b in lst))
BATCH_TO_IDX = {b: idx for idx, b in enumerate(BATCH_LEVELS)}
IDX_TO_BATCH = {idx: b for b, idx in BATCH_TO_IDX.items()}

print("\nGlobal batch levels =", BATCH_LEVELS)
print("N_BATCH_LEVELS =", len(BATCH_LEVELS))

=== Batch candidates per workload ===
llama      -> [1]
gpt2       -> [1]
vgg16      -> [4, 8, 16, 32, 64, 128]
resnet50   -> [4, 8, 16, 32, 64, 128]
vit_base   -> [1, 2, 4, 8, 16, 32, 64]

Global batch levels = [1, 2, 4, 8, 16, 32, 64, 128]
N_BATCH_LEVELS = 8


In [ ]:
def extract_rows_for_workload_batch(spec, batch_value, profile_order):
    """
    Return: dict profile -> stats for this (workload, batch)
    """
    df = pd.read_csv(spec["csv"]).copy()

    mig_col = pick_col(df, ["mig_name"])
    model_col = pick_col(df, ["model"])
    batch_col = pick_col(df, ["batch"])
    peak_mem_col = pick_col(df, ["peak_reserved_mb", "peak_alloc_mb"])
    time_mean_col = pick_col(df, ["time_ms_mean"])

    if mig_col is None or model_col is None or batch_col is None:
        raise ValueError(f"Missing essential columns in {spec['csv']}")

    df["profile"] = df[mig_col].apply(normalize_mig_name)
    df = df[df["profile"].isin(profile_order)].copy()
    df = df[df[model_col] == spec["model_match"]].copy()
    df = df[df[batch_col] == batch_value].copy()

    out = {}

    if spec["family"] == "cv":
        if len(df) == 0:
            return out

        g = df.groupby("profile", as_index=False)[time_mean_col].idxmin()
        chosen = df.loc[g[time_mean_col].values].copy()

        for _, row in chosen.iterrows():
            p = row["profile"]
            latency_ms = float(row[time_mean_col])
            peak_mb = float(row[peak_mem_col]) if peak_mem_col is not None else np.nan

            mem_ok = (peak_mb <= PROFILE_MEM_MB[p]) if not np.isnan(peak_mb) else False
            slo_ok = (latency_ms <= spec["e2e_slo_ms"])
            fit = bool(mem_ok and slo_ok)

            mu = safe_req_rate_from_latency_ms(latency_ms, batch_value) if fit else np.nan

            out[p] = {
                "batch": int(batch_value),
                "latency_ms": latency_ms,
                "peak_mem_mb": peak_mb,
                "ttft_ms": np.nan,
                "tpot_ms": np.nan,
                "service_time_ms": latency_ms,
                "mu_req_per_s": mu,
                "fit_mem": bool(mem_ok),
                "fit_slo": bool(slo_ok),
                "fit": fit,
            }

    elif spec["family"] == "llm":
        prompt_col = pick_col(df, ["prompt_len"])
        output_col = pick_col(df, ["output_tokens"])
        ttft_col = pick_col(df, ["ttft_ms"])
        decode_tps_col = pick_col(df, ["decode_tps"])

        if prompt_col is None or output_col is None or ttft_col is None or decode_tps_col is None:
            raise ValueError(f"Missing LLM columns in {spec['csv']}")

        df = df[df[prompt_col] == spec["prompt_len"]].copy()
        df = df[df[output_col] == spec["output_tokens"]].copy()

        if len(df) == 0:
            return out

        chosen = df.groupby("profile", as_index=False).first()

        for _, row in chosen.iterrows():
            p = row["profile"]
            ttft_ms = float(row[ttft_col])
            decode_tps = float(row[decode_tps_col])
            tpot_ms = decode_tps_to_tpot_ms(decode_tps)
            peak_mb = float(row[peak_mem_col]) if peak_mem_col is not None else np.nan

            mem_ok = (peak_mb <= PROFILE_MEM_MB[p]) if not np.isnan(peak_mb) else False
            slo_ok = (ttft_ms <= spec["ttft_slo_ms"]) and (tpot_ms <= spec["tpot_slo_ms"])
            fit = bool(mem_ok and slo_ok)

            # 这里沿用你之前的近似：service_time = TTFT + output_tokens * TPOT
            service_time_ms = ttft_ms + spec["output_tokens"] * tpot_ms if np.isfinite(tpot_ms) else np.nan
            mu = safe_req_rate_from_latency_ms(service_time_ms, batch_value) if fit else np.nan

            out[p] = {
                "batch": int(batch_value),
                "latency_ms": np.nan,
                "peak_mem_mb": peak_mb,
                "ttft_ms": ttft_ms,
                "tpot_ms": tpot_ms,
                "service_time_ms": service_time_ms,
                "mu_req_per_s": mu,
                "fit_mem": bool(mem_ok),
                "fit_slo": bool(slo_ok),
                "fit": fit,
            }

    else:
        raise ValueError(f"Unknown family: {spec['family']}")

    return out

In [ ]:
def build_workload_batch_profile_tensors(workload_specs, profile_order, batch_levels, batches_per_workload):
    W = len(workload_specs)
    B = len(batch_levels)
    P = len(profile_order)

    mu = np.full((W, B, P), np.nan, dtype=float)
    service_time_ms = np.full((W, B, P), np.nan, dtype=float)
    latency_ms = np.full((W, B, P), np.nan, dtype=float)
    ttft_ms = np.full((W, B, P), np.nan, dtype=float)
    tpot_ms = np.full((W, B, P), np.nan, dtype=float)
    peak_mem_mb = np.full((W, B, P), np.nan, dtype=float)

    fit_mem = np.zeros((W, B, P), dtype=bool)
    fit_slo = np.zeros((W, B, P), dtype=bool)
    fit = np.zeros((W, B, P), dtype=bool)
    batch_allowed = np.zeros((W, B), dtype=bool)

    raw = {}

    for i, spec in enumerate(workload_specs):
        wname = spec["name"]
        raw[wname] = {}

        for batch_value in batches_per_workload[wname]:
            bidx = BATCH_TO_IDX[batch_value]
            batch_allowed[i, bidx] = True

            row_dict = extract_rows_for_workload_batch(spec, batch_value, profile_order)
            raw[wname][batch_value] = row_dict

            for j, p in enumerate(profile_order):
                if p not in row_dict:
                    continue

                d = row_dict[p]
                mu[i, bidx, j] = d["mu_req_per_s"]
                service_time_ms[i, bidx, j] = d["service_time_ms"]
                latency_ms[i, bidx, j] = d["latency_ms"]
                ttft_ms[i, bidx, j] = d["ttft_ms"]
                tpot_ms[i, bidx, j] = d["tpot_ms"]
                peak_mem_mb[i, bidx, j] = d["peak_mem_mb"]
                fit_mem[i, bidx, j] = d["fit_mem"]
                fit_slo[i, bidx, j] = d["fit_slo"]
                fit[i, bidx, j] = d["fit"]

    return {
        "mu": mu,
        "service_time_ms": service_time_ms,
        "latency_ms": latency_ms,
        "ttft_ms": ttft_ms,
        "tpot_ms": tpot_ms,
        "peak_mem_mb": peak_mem_mb,
        "fit_mem": fit_mem,
        "fit_slo": fit_slo,
        "fit": fit,
        "batch_allowed": batch_allowed,
        "raw": raw,
    }

In [ ]:
mat3 = build_workload_batch_profile_tensors(
    workload_specs=WORKLOAD_SPECS,
    profile_order=PROFILE_ORDER,
    batch_levels=BATCH_LEVELS,
    batches_per_workload=BATCHES_PER_WORKLOAD,
)

mu_3d = mat3["mu"]
service_time_ms_3d = mat3["service_time_ms"]
latency_ms_3d = mat3["latency_ms"]
ttft_ms_3d = mat3["ttft_ms"]
tpot_ms_3d = mat3["tpot_ms"]
peak_mem_mb_3d = mat3["peak_mem_mb"]
fit_mem_3d = mat3["fit_mem"]
fit_slo_3d = mat3["fit_slo"]
fit_3d = mat3["fit"]
batch_allowed = mat3["batch_allowed"]

arrival_rate = np.array([w["arrival_rate"] for w in WORKLOAD_SPECS], dtype=float)

print("mu_3d shape =", mu_3d.shape)              # (W, B, P)
print("fit_3d shape =", fit_3d.shape)            # (W, B, P)
print("batch_allowed shape =", batch_allowed.shape)

mu_3d shape = (5, 8, 5)
fit_3d shape = (5, 8, 5)
batch_allowed shape = (5, 8)


In [ ]:
def build_option_table(workload_specs, profile_order, batch_levels, mu_3d, fit_3d,
                       fit_mem_3d, fit_slo_3d, peak_mem_mb_3d, service_time_ms_3d,
                       latency_ms_3d, ttft_ms_3d, tpot_ms_3d):
    rows = []

    for i, spec in enumerate(workload_specs):
        for bidx, b in enumerate(batch_levels):
            for pidx, p in enumerate(profile_order):
                rows.append({
                    "w_idx": i,
                    "workload": spec["name"],
                    "family": spec["family"],
                    "batch_idx": bidx,
                    "batch": int(b),
                    "p_idx": pidx,
                    "profile": p,
                    "mu": mu_3d[i, bidx, pidx],
                    "service_time_ms": service_time_ms_3d[i, bidx, pidx],
                    "latency_ms": latency_ms_3d[i, bidx, pidx],
                    "ttft_ms": ttft_ms_3d[i, bidx, pidx],
                    "tpot_ms": tpot_ms_3d[i, bidx, pidx],
                    "peak_mem_mb": peak_mem_mb_3d[i, bidx, pidx],
                    "fit_mem": bool(fit_mem_3d[i, bidx, pidx]),
                    "fit_slo": bool(fit_slo_3d[i, bidx, pidx]),
                    "fit": bool(fit_3d[i, bidx, pidx]),
                })

    option_df = pd.DataFrame(rows)

    feasible_option_df = option_df[option_df["fit"]].copy().reset_index(drop=True)
    feasible_option_df["opt_idx"] = np.arange(len(feasible_option_df))

    return option_df, feasible_option_df

In [ ]:
option_df, feasible_option_df = build_option_table(
    workload_specs=WORKLOAD_SPECS,
    profile_order=PROFILE_ORDER,
    batch_levels=BATCH_LEVELS,
    mu_3d=mu_3d,
    fit_3d=fit_3d,
    fit_mem_3d=fit_mem_3d,
    fit_slo_3d=fit_slo_3d,
    peak_mem_mb_3d=peak_mem_mb_3d,
    service_time_ms_3d=service_time_ms_3d,
    latency_ms_3d=latency_ms_3d,
    ttft_ms_3d=ttft_ms_3d,
    tpot_ms_3d=tpot_ms_3d,
)

print("All options =", len(option_df))
print("Feasible options =", len(feasible_option_df))

display(feasible_option_df)

All options = 200
Feasible options = 81


,w_idx,workload,family,batch_idx,batch,p_idx,profile,mu,service_time_ms,latency_ms,ttft_ms,tpot_ms,peak_mem_mb,fit_mem,fit_slo,fit,opt_idx
0,0,llama,llm,0,1,0,7g,0.776891,1287.181268,NaN,46.4961,19.385706,6700.0,True,True,True,0
1,0,llama,llm,0,1,1,4g,0.765989,1305.502207,NaN,71.6864,19.278372,6700.0,True,True,True,1
2,0,llama,llm,0,1,2,3g,0.740961,1349.599316,NaN,92.8640,19.636489,6700.0,True,True,True,2
3,1,gpt2,llm,0,1,0,7g,1.156114,864.966308,NaN,15.1986,13.277620,782.0,True,True,True,3
4,1,gpt2,llm,0,1,1,4g,1.122924,890.532504,NaN,15.5701,13.671288,782.0,True,True,True,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,4,vit_base,cv,5,32,2,3g,1356.794573,23.585000,23.5850,NaN,NaN,626.0,True,True,True,76
77,4,vit_base,cv,5,32,3,2g,857.524915,37.316700,37.3167,NaN,NaN,626.0,True,True,True,77
78,4,vit_base,cv,6,64,0,7g,2676.491621,23.911900,23.9119,NaN,NaN,1060.0,True,True,True,78
79,4,vit_base,cv,6,64,1,4g,1718.983758,37.231300,37.2313,NaN,NaN,1060.0,True,True,True,79


In [ ]:
FEASIBLE_OPTIONS_BY_WORKLOAD = {
    i: feasible_option_df[feasible_option_df["w_idx"] == i]["opt_idx"].tolist()
    for i in range(N_WORKLOADS)
}

FEASIBLE_OPTIONS_BY_PROFILE = {
    j: feasible_option_df[feasible_option_df["p_idx"] == j]["opt_idx"].tolist()
    for j in range(len(PROFILE_ORDER))
}

print("=== #feasible options by workload ===")
for i, w in enumerate(WORKLOAD_NAMES):
    print(f"{w:10s} -> {len(FEASIBLE_OPTIONS_BY_WORKLOAD[i])}")

=== #feasible options by workload ===
llama      -> 3
gpt2       -> 5
vgg16      -> 18
resnet50   -> 23
vit_base   -> 32


In [ ]:
def show_feasible_options_per_workload(feasible_option_df):
    rows = []
    for w, g in feasible_option_df.groupby("workload"):
        items = []
        for _, r in g.sort_values(["batch", "profile"]).iterrows():
            items.append(f"(b={int(r['batch'])}, p={r['profile']}, mu={r['mu']:.4f})")
        rows.append([w, len(g), "; ".join(items[:20]) + (" ..." if len(items) > 20 else "")])

    print(tabulate(
        rows,
        headers=["Workload", "#Feasible (batch,profile) options", "Options"],
        tablefmt="github"
    ))

show_feasible_options_per_workload(feasible_option_df)

| Workload   |   #Feasible (batch,profile) options | Options                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
|------------|-------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
def check_global_feasibility_batch(workload_specs, feasible_option_df):
    ok = True
    rows = []

    for i, spec in enumerate(workload_specs):
        g = feasible_option_df[feasible_option_df["w_idx"] == i].copy()
        pairs = list(zip(g["batch"].tolist(), g["profile"].tolist()))
        rows.append([
            spec["name"],
            len(pairs),
            pairs[:15] if len(pairs) <= 15 else pairs[:15] + ["..."]
        ])
        if len(pairs) == 0:
            ok = False

    print(tabulate(
        rows,
        headers=["Workload", "#Feasible (batch,profile)", "Feasible pairs"],
        tablefmt="github"
    ))

    if ok:
        print("\n[OK] Every workload has at least one feasible (batch, profile) option.")
    else:
        print("\n[WARN] Some workload has no feasible (batch, profile) option. MILP will be infeasible.")

check_global_feasibility_batch(WORKLOAD_SPECS, feasible_option_df)

| Workload   |   #Feasible (batch,profile) | Feasible pairs                                                                                                                                                                    |
|------------|-----------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| llama      |                           3 | [(1, '7g'), (1, '4g'), (1, '3g')]                                                                                                                                                 |
| gpt2       |                           5 | [(1, '7g'), (1, '4g'), (1, '3g'), (1, '2g'), (1, '1g')]                                                                                                                           |
| vgg16      |                          18 | [(4, '7g'), (4, '4g'), (4, '3g'), (4, '2g'), (4, '1g'),

In [ ]:
def print_experiment_header_batch():
    print("=" * 90)
    print("Simulation: Min #GPU under arrival-rate / SLO / MIG constraints (batch-selectable)")
    print("=" * 90)
    print(f"Workloads      : {WORKLOAD_NAMES}")
    print(f"Profiles       : {PROFILE_ORDER}")
    print(f"Batch levels   : {BATCH_LEVELS}")
    print(f"#Templates     : {len(TEMPLATES)}")
    print()

    rows = []
    for w in WORKLOAD_SPECS:
        if w["family"] == "llm":
            slo_str = f"TTFT<={w['ttft_slo_ms']}ms, TPOT<={w['tpot_slo_ms']}ms"
            extra = f"prompt={w['prompt_len']}, out={w['output_tokens']}"
        else:
            slo_str = f"E2E<={w['e2e_slo_ms']}ms"
            extra = "-"

        rows.append([
            w["name"],
            w["family"],
            BATCHES_PER_WORKLOAD[w["name"]],
            w["arrival_rate"],
            slo_str,
            extra
        ])

    print(tabulate(
        rows,
        headers=["Workload", "Family", "Batch candidates", "Arrival(req/s)", "SLO", "Extra"],
        tablefmt="github",
        floatfmt=".4f"
    ))

print_experiment_header_batch()

Simulation: Min #GPU under arrival-rate / SLO / MIG constraints (batch-selectable)
Workloads      : ['llama', 'gpt2', 'vgg16', 'resnet50', 'vit_base']
Profiles       : ['7g', '4g', '3g', '2g', '1g']
Batch levels   : [1, 2, 4, 8, 16, 32, 64, 128]
#Templates     : 12

| Workload   | Family   | Batch candidates         |   Arrival(req/s) | SLO                         | Extra               |
|------------|----------|--------------------------|------------------|-----------------------------|---------------------|
| llama      | llm      | [1]                      |           3.0000 | TTFT<=100.0ms, TPOT<=25.0ms | prompt=1024, out=64 |
| gpt2       | llm      | [1]                      |         100.0000 | TTFT<=20.0ms, TPOT<=15.0ms  | prompt=64, out=64   |
| vgg16      | cv       | [4, 8, 16, 32, 64, 128]  |        3000.0000 | E2E<=50.0ms                 | -                   |
| resnet50   | cv       | [4, 8, 16, 32, 64, 128]  |        3000.0000 | E2E<=50.0ms                 | -          

In [ ]:
def format_instance_list(instances):
    items = []
    for inst in instances:
        items.append(f"({inst['batch']}, {inst['profile']}, {inst['count']})")
    return "; ".join(items)

def print_solution_summary(
    method_name,
    elapsed,
    feasible,
    objective,
    K_total,
    templates_used,
    alloc
):
    print("=" * 90)
    print(f"Method        : {method_name}")
    print(f"Elapsed (s)   : {elapsed:.4f}")
    print(f"Feasible      : {feasible}")
    #print(f"GPU count     : {sum(K_total.values())}")
    print(f"GPU count     : {len(templates_used)}")
    print(f"Objective     : {objective:.4f}" if objective is not None else "Objective     : -")
    print(f"Templates     : {templates_used}")
    print(f"K_total       : {K_total}")
    print("=" * 90)


    rows = []
    for w in alloc:
        ratio = w["provided"] / w["arrival"] if w["arrival"] > 0 else 0.0

        rows.append([
            w["workload"],
            f"{w['arrival']:.4f}",
            f"{w['provided']:.4f}",
            f"{ratio:.3f}",
            f"{w['slack']:.4f}",
            format_instance_list(w["instances"])
        ])

    print("\nPer-workload allocation / rate summary:")
    print(tabulate(
        rows,
        headers=[
            "Workload",
            "Arrival",
            "Provided",
            "Provided/Arrival",
            "Slack",
            "Placement (batch,profile,count)"
        ],
        tablefmt="github"
    ))

In [ ]:
def build_allocation_from_x(
    feasible_option_df,
    x_sol,
    arrival_rate
):
    """
    x_sol: dict {opt_idx: value}
    """

    alloc = []

    for i in range(len(arrival_rate)):
        g = feasible_option_df[feasible_option_df["w_idx"] == i]

        instances = []
        provided = 0.0

        for _, row in g.iterrows():
            o = row["opt_idx"]
            x_val = x_sol.get(o, 0)

            if x_val <= 0:
                continue

            mu = row["mu"]
            provided += x_val * mu

            instances.append({
                "batch": int(row["batch"]),
                "profile": row["profile"],
                "count": int(x_val),
                "mu": mu
            })

        arrival = arrival_rate[i]
        slack = provided - arrival

        alloc.append({
            "workload": g["workload"].iloc[0],
            "arrival": arrival,
            "provided": provided,
            "slack": slack,
            "instances": instances
        })

    return alloc

In [ ]:
def compute_slo_violation_rate(alloc):
    rows = []

    for w in alloc:
        if w["arrival"] <= 0:
            viol = 0.0
        else:
            viol = max(0.0, 1.0 - w["provided"] / w["arrival"])

        rows.append([
            w["workload"],
            f"{viol:.4f}"
        ])

    print("\nSLO Violation Rate per workload:")
    print(tabulate(
        rows,
        headers=["Workload", "Violation Rate"],
        tablefmt="github"
    ))

# One Workload Test

In [ ]:
SINGLE_WORKLOAD_NAME = "vgg16"
# SINGLE_WORKLOAD_NAME = "llama"
# SINGLE_WORKLOAD_NAME = "gpt2"
# SINGLE_WORKLOAD_NAME = "resnet50"
# SINGLE_WORKLOAD_NAME = "vit_base"

single_w_idx = WORKLOAD_NAMES.index(SINGLE_WORKLOAD_NAME)
single_arrival = arrival_rate[single_w_idx]

print("Single workload =", SINGLE_WORKLOAD_NAME)
print("Workload index  =", single_w_idx)
print("Arrival rate    =", single_arrival)

In [ ]:
single_option_df = feasible_option_df[
    feasible_option_df["w_idx"] == single_w_idx
].copy().reset_index(drop=True)

single_option_df["local_opt_idx"] = np.arange(len(single_option_df))

In [ ]:
from ortools.linear_solver import pywraplp

def solve_single_workload_milp_min_gpu(
    single_option_df,
    single_arrival,
    templates,
    template_K,
    profile_order,
    time_limit_sec=60,
    verbose=True
):
    start = time.time()

    solver = pywraplp.Solver.CreateSolver("CBC")
    if solver is None:
        raise RuntimeError("CBC solver is not available.")

    solver.SetTimeLimit(int(time_limit_sec * 1000))

    n_opts = len(single_option_df)
    n_templates = len(templates)

    # -------------------------------------------------
    # Variables
    # x[o] = number of instances using option o=(batch, profile)
    # y[t] = number of GPUs using template t
    # -------------------------------------------------
    x = {}
    for o in range(n_opts):
        x[o] = solver.IntVar(0, solver.infinity(), f"x_{o}")

    y = {}
    for t in range(n_templates):
        y[t] = solver.IntVar(0, solver.infinity(), f"y_{t}")

    # -------------------------------------------------
    # Constraint 1: provided throughput >= arrival rate
    # -------------------------------------------------
    c_arrival = solver.RowConstraint(single_arrival, solver.infinity(), "arrival")
    for o in range(n_opts):
        mu_o = float(single_option_df.loc[o, "mu"])
        c_arrival.SetCoefficient(x[o], mu_o)

    # -------------------------------------------------
    # Constraint 2: per-profile usage <= capacity from chosen templates
    # -------------------------------------------------
    profile_to_opts = {}
    for p in profile_order:
        profile_to_opts[p] = single_option_df[
            single_option_df["profile"] == p
        ]["local_opt_idx"].tolist()

    for p in profile_order:
        ct = solver.RowConstraint(-solver.infinity(), 0.0, f"profile_cap_{p}")

        # LHS: used instances on profile p
        for o in profile_to_opts[p]:
            ct.SetCoefficient(x[o], 1.0)

        # RHS moved to LHS:
        # used - sum_t K[t,p] y[t] <= 0
        for t in range(n_templates):
            ct.SetCoefficient(y[t], -float(template_K[t][p]))

    # -------------------------------------------------
    # Objective: min number of GPUs
    # -------------------------------------------------
    obj = solver.Objective()
    for t in range(n_templates):
        obj.SetCoefficient(y[t], 1.0)
    obj.SetMinimization()

    # -------------------------------------------------
    # Solve
    # -------------------------------------------------
    status = solver.Solve()
    elapsed = time.time() - start

    feasible = status in (
        pywraplp.Solver.OPTIMAL,
        pywraplp.Solver.FEASIBLE
    )

    if not feasible:
        return {
            "method": "MILP-single-batch",
            "feasible": False,
            "status": status,
            "elapsed": elapsed,
            "objective": None,
            "x_sol": {},
            "y_sol": {},
            "provided": 0.0,
            "single_option_df": single_option_df,
        }

    x_sol = {}
    for o in range(n_opts):
        val = x[o].solution_value()
        if val > 1e-9:
            x_sol[o] = int(round(val))

    y_sol = {}
    for t in range(n_templates):
        val = y[t].solution_value()
        if val > 1e-9:
            y_sol[t] = int(round(val))

    provided = 0.0
    for o, xv in x_sol.items():
        provided += xv * float(single_option_df.loc[o, "mu"])

    objective = obj.Value()

    if verbose:
        print(f"[MILP-single] status   = {status}")
        print(f"[MILP-single] feasible = {feasible}")
        print(f"[MILP-single] elapsed  = {elapsed:.4f}s")
        print(f"[MILP-single] obj      = {objective:.4f}")
        print(f"[MILP-single] arrival  = {single_arrival:.6f}")
        print(f"[MILP-single] provided = {provided:.6f}")

    return {
        "method": "MILP-single-batch",
        "feasible": True,
        "status": status,
        "elapsed": elapsed,
        "objective": objective,
        "x_sol": x_sol,
        "y_sol": y_sol,
        "provided": provided,
        "single_option_df": single_option_df,
    }

In [ ]:
def build_single_alloc_from_x(single_option_df, x_sol, single_arrival):
    instances = []
    provided = 0.0

    for o, xv in sorted(x_sol.items()):
        row = single_option_df.loc[o]
        mu_o = float(row["mu"])
        provided += xv * mu_o

        instances.append({
            "batch": int(row["batch"]),
            "profile": row["profile"],
            "count": int(xv),
            "mu": mu_o
        })

    slack = provided - single_arrival

    alloc = [{
        "workload": single_option_df["workload"].iloc[0] if len(single_option_df) > 0 else "unknown",
        "arrival": float(single_arrival),
        "provided": float(provided),
        "slack": float(slack),
        "instances": instances
    }]

    return alloc

In [ ]:
def build_template_summary_from_y(y_sol, templates, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    templates_used = []

    for t, cnt in sorted(y_sol.items()):
        if cnt <= 0:
            continue

        tname = templates[t][0]
        for _ in range(cnt):
            templates_used.append(tname)

        for p in profile_order:
            K_total[p] += int(template_K[t][p]) * int(cnt)

    return K_total, templates_used

In [ ]:
def print_single_workload_solution(
    result,
    single_arrival,
    templates,
    template_K,
    profile_order
):
    if not result["feasible"]:
        print("=" * 90)
        print(f"Method        : {result['method']}")
        print(f"Elapsed (s)   : {result['elapsed']:.4f}")
        print("Feasible      : False")
        print("=" * 90)
        return

    K_total, templates_used = build_template_summary_from_y(
        result["y_sol"], templates, template_K, profile_order
    )

    alloc = build_single_alloc_from_x(
        result["single_option_df"],
        result["x_sol"],
        single_arrival
    )

    print_solution_summary(
        method_name=result["method"],
        elapsed=result["elapsed"],
        feasible=result["feasible"],
        objective=result["objective"],
        K_total=K_total,
        templates_used=templates_used,
        alloc=alloc
    )

    compute_slo_violation_rate(alloc)

In [ ]:
def print_single_instance_details(single_option_df, x_sol):
    rows = []
    total_mu = 0.0

    for o, xv in sorted(x_sol.items()):
        row = single_option_df.loc[o]
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal

        rows.append([
            int(o),
            row["workload"],
            int(row["batch"]),
            row["profile"],
            int(xv),
            f"{mu_o:.6f}",
            f"{subtotal:.6f}"
        ])

    print("\nChosen instances:")
    print(tabulate(
        rows,
        headers=["opt", "workload", "batch", "profile", "count", "mu", "count*mu"],
        tablefmt="github"
    ))
    print(f"\nTotal provided throughput = {total_mu:.6f}")

# single result

In [ ]:
single_result = solve_single_workload_milp_min_gpu(
    single_option_df=single_option_df,
    single_arrival=single_arrival,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    time_limit_sec=60,
    verbose=True
)

print_single_workload_solution(
    result=single_result,
    single_arrival=single_arrival,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER
)

if single_result["feasible"]:
    print_single_instance_details(
        single_option_df=single_result["single_option_df"],
        x_sol=single_result["x_sol"]
    )

# Enumerate

In [ ]:
# =========================================================
# Oracle for nofixbs (rewrite in old-oracle style)
# - enumerate G
# - enumerate all template multisets at this G
# - solve exact placement for each K_total
# - keep best solutions at this G
# - return once this G has any feasible solution
# =========================================================

from ortools.sat.python import cp_model
import numpy as np
import math
import time
from tabulate import tabulate


# ---------------------------------------------------------
# helper 1
# ---------------------------------------------------------
def template_count_vec_to_K_total(tcount_vec, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    for t_idx, cnt in enumerate(tcount_vec):
        if cnt == 0:
            continue
        for p in profile_order:
            K_total[p] += int(cnt) * int(template_K[t_idx][p])
    return K_total


# ---------------------------------------------------------
# helper 2
# ---------------------------------------------------------
def expand_template_count_vec(tcount_vec, templates):
    out = []
    for t_idx, cnt in enumerate(tcount_vec):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


# ---------------------------------------------------------
# helper 3
# ---------------------------------------------------------
def enumerate_template_count_vectors(G, nT):
    vec = [0] * nT

    def dfs(pos, remain):
        if pos == nT - 1:
            vec[pos] = remain
            yield tuple(vec)
            return
        for x in range(remain + 1):
            vec[pos] = x
            yield from dfs(pos + 1, remain - x)

    yield from dfs(0, G)


# ---------------------------------------------------------
# helper 4
# exact solver for fixed K_total
#
# mode:
#   "first_feasible"   -> just find a feasible solution
#   "closest"          -> min total slack
#   "least_profiles"   -> min used profile types
# ---------------------------------------------------------
def solve_exact_placement_given_K_batch(
    feasible_option_df,
    arrival_rate,
    profile_order,
    K_total,
    n_workloads,
    mode="first_feasible",
    time_limit_s=10.0,
    rate_scale=10000,
    verbose=False,
):
    df = feasible_option_df.copy().reset_index(drop=True)
    start = time.time()

    model = cp_model.CpModel()
    n_opts = len(df)
    opt_ids = list(range(n_opts))

    options_by_workload = {i: [] for i in range(n_workloads)}
    options_by_profile = {p: [] for p in profile_order}

    for o in opt_ids:
        w_idx = int(df.loc[o, "w_idx"])
        p = df.loc[o, "profile"]
        options_by_workload[w_idx].append(o)
        options_by_profile[p].append(o)

    # sanity
    for i in range(n_workloads):
        if len(options_by_workload[i]) == 0:
            return {
                "feasible": False,
                "x_sol": None,
                "alloc": None,
                "provided_rate": None,
                "total_slack": None,
                "total_instances": None,
                "used_profile_types": None,
                "solver_status": "NO_OPTION_FOR_WORKLOAD",
            }

    # x[o]
    x = {}
    for o in opt_ids:
        p = df.loc[o, "profile"]

        if K_total[p] <= 0:
            x[o] = model.NewIntVar(0, 0, f"x_{o}")
            continue

        mu_o = float(df.loc[o, "mu"])
        w_idx = int(df.loc[o, "w_idx"])

        ub_by_cap = int(K_total[p])
        ub_by_demand = int(math.ceil(arrival_rate[w_idx] / mu_o)) if mu_o > 0 else 0
        ub = max(0, min(ub_by_cap, ub_by_demand))

        x[o] = model.NewIntVar(0, ub, f"x_{o}")

    # throughput constraints
    slack_vars = []
    provided_scaled_exprs = []

    for i in range(n_workloads):
        expr_terms = []
        for o in options_by_workload[i]:
            mu_o = float(df.loc[o, "mu"])
            mu_scaled = int(round(mu_o * rate_scale))
            expr_terms.append(mu_scaled * x[o])

        provided_i = sum(expr_terms)
        demand_i = int(math.ceil(float(arrival_rate[i]) * rate_scale))

        model.Add(provided_i >= demand_i)

        slack_i = model.NewIntVar(0, 10**12, f"slack_{i}")
        model.Add(provided_i - demand_i == slack_i)

        provided_scaled_exprs.append(provided_i)
        slack_vars.append(slack_i)

    # profile capacity constraints
    for p in profile_order:
        model.Add(sum(x[o] for o in options_by_profile[p]) <= int(K_total[p]))

    total_instances = model.NewIntVar(0, sum(K_total.values()), "total_instances")
    model.Add(total_instances == sum(x[o] for o in opt_ids))

    total_slack = model.NewIntVar(0, 10**12, "total_slack")
    model.Add(total_slack == sum(slack_vars))

    # used profile types
    y = {}
    for p in profile_order:
        y[p] = model.NewBoolVar(f"y_{p}")
        model.Add(sum(x[o] for o in options_by_profile[p]) <= int(K_total[p]) * y[p])

    used_profile_types = model.NewIntVar(0, len(profile_order), "used_profile_types")
    model.Add(used_profile_types == sum(y[p] for p in profile_order))

    # objective
    if mode == "first_feasible":
        pass
    elif mode == "closest":
        # primary: min total slack
        # secondary: min total instances
        BIG = 10**6
        model.Minimize(BIG * total_slack + total_instances)
    elif mode == "least_profiles":
        # primary: min used profile types
        # secondary: min total_slack
        # tertiary: min total_instances
        BIG1 = 10**9
        BIG2 = 10**4
        model.Minimize(BIG1 * used_profile_types + BIG2 * total_slack + total_instances)
    elif mode == "least_instances":
        # primary: min total_instances
        # secondary: min total_slack
        # tertiary: min used_profile_types
        BIG1 = 10**9
        BIG2 = 10**4
        model.Minimize(BIG1 * total_instances + BIG2 * total_slack + used_profile_types)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit_s)
    solver.parameters.num_search_workers = 8
    solver.parameters.log_search_progress = verbose

    status = solver.Solve(model)
    feasible = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)

    if not feasible:
        return {
            "feasible": False,
            "x_sol": None,
            "alloc": None,
            "provided_rate": None,
            "total_slack": None,
            "total_instances": None,
            "used_profile_types": None,
            "solver_status": solver.StatusName(status),
        }

    # extract
    x_sol = {}
    for o in opt_ids:
        val = solver.Value(x[o])
        if val > 0:
            x_sol[int(df.loc[o, "opt_idx"])] = int(val)

    alloc = build_allocation_from_x(
        feasible_option_df=feasible_option_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )

    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)

    return {
        "feasible": True,
        "x_sol": x_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": float(sum(w["slack"] for w in alloc)),
        "total_instances": int(sum(x_sol.values())),
        "used_profile_types": int(sum(
            1 for p in profile_order
            if any(inst["profile"] == p for w in alloc for inst in w["instances"])
        )),
        "solver_status": solver.StatusName(status),
    }


# ---------------------------------------------------------
# score helpers
# ---------------------------------------------------------
def score_closest(sol):
    return (sol["total_slack"], sol["total_instances"])


def score_least_profiles(sol):
    return (sol["used_profile_types"], sol["total_slack"], sol["total_instances"])


def score_first_feasible(sol):
    return (sol["total_instances"], sol["used_profile_types"])

def score_least_instances(sol):
    return (sol["total_instances"], sol["total_slack"], sol["used_profile_types"])


# ---------------------------------------------------------
# main oracle
# ---------------------------------------------------------
def oracle_min_gpu_batch_oldstyle(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    max_gpus=8,
    per_combo_time_limit_s=5.0,
    verbose=False,
):
    t0 = time.time()
    nT = len(templates)
    explored_combos = 0

    best_first_at_G = None
    best_closest_at_G = None
    best_profiles_at_G = None
    best_instances_at_G = None

    for G in range(1, max_gpus + 1):
        if verbose:
            print(f"[Oracle-batch] Try G = {G}")

        best_first_at_G = None
        best_closest_at_G = None
        best_profiles_at_G = None
        best_instances_at_G = None

        for tcount_vec in enumerate_template_count_vectors(G, nT):
            explored_combos += 1
            K_total = template_count_vec_to_K_total(tcount_vec, template_K, profile_order)
            template_list = expand_template_count_vec(tcount_vec, templates)

            # 1) first feasible / baseline
            sol_ff = solve_exact_placement_given_K_batch(
                feasible_option_df=feasible_option_df,
                arrival_rate=arrival_rate,
                profile_order=profile_order,
                K_total=K_total,
                n_workloads=n_workloads,
                mode="first_feasible",
                time_limit_s=per_combo_time_limit_s,
                verbose=False,
            )

            if not sol_ff["feasible"]:
                continue

            cand_ff = {
                "method": "Oracle-batch(first-feasible@minG)",
                "feasible": True,
                "gpu_count": G,
                "objective": G,
                "chosen_templates": template_list,
                "template_count_vec": tcount_vec,
                "K_total": K_total,
                "x_sol": sol_ff["x_sol"],
                "alloc": sol_ff["alloc"],
                "provided_rate": sol_ff["provided_rate"],
                "total_slack": sol_ff["total_slack"],
                "total_instances": sol_ff["total_instances"],
                "used_profile_types": sol_ff["used_profile_types"],
            }

            if (best_first_at_G is None) or (score_first_feasible(cand_ff) < score_first_feasible(best_first_at_G)):
                best_first_at_G = cand_ff

            # 2) closest
            sol_close = solve_exact_placement_given_K_batch(
                feasible_option_df=feasible_option_df,
                arrival_rate=arrival_rate,
                profile_order=profile_order,
                K_total=K_total,
                n_workloads=n_workloads,
                mode="closest",
                time_limit_s=per_combo_time_limit_s,
                verbose=False,
            )

            if sol_close["feasible"]:
                cand_close = {
                    "method": "Oracle-batch(min-slack@minG)",
                    "feasible": True,
                    "gpu_count": G,
                    "objective": G,
                    "chosen_templates": template_list,
                    "template_count_vec": tcount_vec,
                    "K_total": K_total,
                    "x_sol": sol_close["x_sol"],
                    "alloc": sol_close["alloc"],
                    "provided_rate": sol_close["provided_rate"],
                    "total_slack": sol_close["total_slack"],
                    "total_instances": sol_close["total_instances"],
                    "used_profile_types": sol_close["used_profile_types"],
                }

                if (best_closest_at_G is None) or (score_closest(cand_close) < score_closest(best_closest_at_G)):
                    best_closest_at_G = cand_close

            # 3) least profiles
            sol_prof = solve_exact_placement_given_K_batch(
                feasible_option_df=feasible_option_df,
                arrival_rate=arrival_rate,
                profile_order=profile_order,
                K_total=K_total,
                n_workloads=n_workloads,
                mode="least_profiles",
                time_limit_s=per_combo_time_limit_s,
                verbose=False,
            )

            if sol_prof["feasible"]:
                cand_prof = {
                    "method": "Oracle-batch(min-profile-types@minG)",
                    "feasible": True,
                    "gpu_count": G,
                    "objective": G,
                    "chosen_templates": template_list,
                    "template_count_vec": tcount_vec,
                    "K_total": K_total,
                    "x_sol": sol_prof["x_sol"],
                    "alloc": sol_prof["alloc"],
                    "provided_rate": sol_prof["provided_rate"],
                    "total_slack": sol_prof["total_slack"],
                    "total_instances": sol_prof["total_instances"],
                    "used_profile_types": sol_prof["used_profile_types"],
                }

                if (best_profiles_at_G is None) or (score_least_profiles(cand_prof) < score_least_profiles(best_profiles_at_G)):
                    best_profiles_at_G = cand_prof


                    # 4) least instances
            sol_inst = solve_exact_placement_given_K_batch(
                feasible_option_df=feasible_option_df,
                arrival_rate=arrival_rate,
                profile_order=profile_order,
                K_total=K_total,
                n_workloads=n_workloads,
                mode="least_instances",
                time_limit_s=per_combo_time_limit_s,
                verbose=False,
            )

            if sol_inst["feasible"]:
                cand_inst = {
                    "method": "Oracle-batch(min-instances@minG)",
                    "feasible": True,
                    "gpu_count": G,
                    "objective": G,
                    "chosen_templates": template_list,
                    "template_count_vec": tcount_vec,
                    "K_total": K_total,
                    "x_sol": sol_inst["x_sol"],
                    "alloc": sol_inst["alloc"],
                    "provided_rate": sol_inst["provided_rate"],
                    "total_slack": sol_inst["total_slack"],
                    "total_instances": sol_inst["total_instances"],
                    "used_profile_types": sol_inst["used_profile_types"],
                }

                if (best_instances_at_G is None) or (score_least_instances(cand_inst) < score_least_instances(best_instances_at_G)):
                    best_instances_at_G = cand_inst



        # exactly like old oracle:
        # first feasible G is globally optimal in #GPU
        if best_first_at_G is not None:
            elapsed = time.time() - t0
            best_first_at_G["elapsed"] = elapsed
            best_closest_at_G["elapsed"] = elapsed if best_closest_at_G is not None else elapsed
            best_profiles_at_G["elapsed"] = elapsed if best_profiles_at_G is not None else elapsed


            if best_instances_at_G is not None:
                best_instances_at_G["elapsed"] = elapsed

            return {
                "method": "Oracle-batch(oldstyle)",
                "elapsed": elapsed,
                "feasible": True,
                "gpu_count": G,
                "objective": G,
                "first_feasible_result": best_first_at_G,
                "closest_result": best_closest_at_G,
                "least_profile_result": best_profiles_at_G,
                "least_instance_result": best_instances_at_G,
                "explored_combos": explored_combos,
            }

    return {
        "method": "Oracle-batch(oldstyle)",
        "elapsed": time.time() - t0,
        "feasible": False,
        "gpu_count": None,
        "objective": None,
        "first_feasible_result": None,
        "closest_result": None,
        "least_profile_result": None,
        "least_instance_result": None,
        "explored_combos": explored_combos,
    }


# ---------------------------------------------------------
# printer
# ---------------------------------------------------------
def print_oracle_batch_oldstyle_result(res):
    print("=" * 90)
    print(f"Method         : {res['method']}")
    print(f"Elapsed (s)    : {res['elapsed']:.4f}")
    print(f"Feasible       : {res['feasible']}")
    print(f"GPU count      : {res['gpu_count']}")
    print(f"Explored combos: {res['explored_combos']}")
    print("=" * 90)

    for title, sol in [
        ("First feasible / old-style tie-break", res["first_feasible_result"]),
        ("Closest throughput to arrival", res["closest_result"]),
        ("Least used profile types", res["least_profile_result"]),
        ("Least used profile instances", res["least_instance_result"]),
    ]:
        print("\n" + "#" * 90)
        print(title)
        print("#" * 90)

        if sol is None:
            print("No solution.")
            continue

        print_solution_summary(
            method_name=sol["method"],
            elapsed=sol["elapsed"],
            feasible=sol["feasible"],
            objective=sol["objective"],
            K_total=sol["K_total"],
            templates_used=sol["chosen_templates"],
            alloc=sol["alloc"]
        )
        print(f"\nTotal slack        : {sol['total_slack']:.6f}")
        print(f"Total instances    : {sol['total_instances']}")
        print(f"Used profile types : {sol['used_profile_types']}")
        compute_slo_violation_rate(sol["alloc"])

# enumerate result

In [ ]:
oracle_res = oracle_min_gpu_batch_oldstyle(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    max_gpus=8,
    per_combo_time_limit_s=5.0,
    verbose=True,
)

print_oracle_batch_oldstyle_result(oracle_res)

# MILP

In [ ]:
!pip install gurobipy

In [ ]:
import time
import math
import numpy as np
from tabulate import tabulate
import gurobipy as gp
from gurobipy import GRB


def milp_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def milp_build_K_total(y_sol, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            K_total[p] += int(cnt) * int(template_K[t_idx][p])
    return K_total


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0

    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal

        rows.append([
            o,
            row["workload"],
            int(row["batch"]),
            row["profile"],
            xv,
            f"{mu_o:.6f}",
            f"{subtotal:.6f}",
        ])

    print("\nChosen instances:")
    print(tabulate(
        rows,
        headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"],
        tablefmt="github"
    ))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


def solve_milp_gurobi_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    verbose=False,
):
    start = time.time()

    df = feasible_option_df.copy().reset_index(drop=True)
    n_opts = len(df)
    nT = len(templates)

    opt_rows = list(range(n_opts))
    t_ids = list(range(nT))

    options_by_workload = {i: [] for i in range(n_workloads)}
    options_by_profile = {p: [] for p in profile_order}

    for r in opt_rows:
        w_idx = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        options_by_workload[w_idx].append(r)
        options_by_profile[p].append(r)

    for i in range(n_workloads):
        if len(options_by_workload[i]) == 0:
            return {
                "method": "MILP-Gurobi(batch)",
                "feasible": False,
                "status": f"NO_OPTION_FOR_WORKLOAD_{i}",
                "elapsed": time.time() - start,
            }

    model = gp.Model("milp_batch_nofixbs")

    if not verbose:
        model.Params.OutputFlag = 0
    if time_limit_s is not None:
        model.Params.TimeLimit = float(time_limit_s)
    if mip_gap is not None:
        model.Params.MIPGap = float(mip_gap)
    if threads is not None:
        model.Params.Threads = int(threads)

    # -----------------------------
    # Variables
    # -----------------------------
    y = model.addVars(t_ids, vtype=GRB.INTEGER, lb=0, name="y")

    total_gpu = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_gpu")
    model.addConstr(total_gpu == gp.quicksum(y[t] for t in t_ids), name="def_total_gpu")

    cap = {}
    for p in profile_order:
        cap[p] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"cap_{p}")
        model.addConstr(
            cap[p] == gp.quicksum(int(template_K[t][p]) * y[t] for t in t_ids),
            name=f"def_cap_{p}"
        )

    # loose upper bound on total GPU count, computed once
    ub_gpu_loose = 0
    for i in range(n_workloads):
        g = df[df["w_idx"] == i]
        best_mu = float(g["mu"].max())
        ub_gpu_loose += int(math.ceil(arrival_rate[i] / best_mu)) if best_mu > 0 else 0
    ub_gpu_loose = max(1, ub_gpu_loose)

    x = {}
    for r in opt_rows:
        w_idx = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        mu_o = float(df.loc[r, "mu"])

        ub_by_demand = int(math.ceil(arrival_rate[w_idx] / mu_o)) + 5 if mu_o > 0 else 0
        max_profile_per_gpu = max(int(template_K[t][p]) for t in t_ids)
        ub_by_gpu = ub_gpu_loose * max_profile_per_gpu
        ub = max(1, min(ub_by_demand, ub_by_gpu))

        x[r] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=ub, name=f"x_{r}")

    provided = {}
    slack = {}
    for i in range(n_workloads):
        provided[i] = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name=f"provided_{i}")
        slack[i] = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name=f"slack_{i}")

        expr = gp.quicksum(float(df.loc[r, "mu"]) * x[r] for r in options_by_workload[i])

        model.addConstr(provided[i] == expr, name=f"def_provided_{i}")
        model.addConstr(provided[i] >= float(arrival_rate[i]), name=f"demand_{i}")
        model.addConstr(slack[i] == provided[i] - float(arrival_rate[i]), name=f"def_slack_{i}")

    for p in profile_order:
        model.addConstr(
            gp.quicksum(x[r] for r in options_by_profile[p]) <= cap[p],
            name=f"profile_cap_{p}"
        )

    total_slack = model.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="total_slack")
    model.addConstr(
        total_slack == gp.quicksum(slack[i] for i in range(n_workloads)),
        name="def_total_slack"
    )

    total_instances = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_instances")
    model.addConstr(
        total_instances == gp.quicksum(x[r] for r in opt_rows),
        name="def_total_instances"
    )

    total_remaining_slots = model.addVar(vtype=GRB.INTEGER, lb=0, name="total_remaining_slots")
    model.addConstr(
        total_remaining_slots ==
        gp.quicksum(cap[p] for p in profile_order) - gp.quicksum(x[r] for r in opt_rows),
        name="def_total_remaining_slots"
    )

    # -----------------------------
    # Multi-objective
    # -----------------------------
    model.ModelSense = GRB.MINIMIZE
    model.setObjectiveN(total_gpu,       index=0, priority=3, weight=1.0, name="obj_gpu")
    model.setObjectiveN(total_slack,     index=1, priority=2, weight=1.0, name="obj_slack")
    #model.setObjectiveN(total_instances, index=2, priority=1, weight=1.0, name="obj_inst")
    model.setObjectiveN(total_remaining_slots, index=2, priority=1, weight=-1.0, name="obj_remaining")
    model.optimize()

    elapsed = time.time() - start
    feasible = model.SolCount > 0

    status_map = {
        GRB.OPTIMAL: "OPTIMAL",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.INTERRUPTED: "INTERRUPTED",
    }
    status_str = status_map.get(model.Status, str(model.Status))

    if not feasible:
        return {
            "method": "MILP-Gurobi(batch)",
            "feasible": False,
            "status": status_str,
            "elapsed": elapsed,
        }

    y_sol = {}
    for t in t_ids:
        v = int(round(y[t].X))
        if v > 0:
            y_sol[t] = v

    x_sol = {}
    for r in opt_rows:
        v = int(round(x[r].X))
        if v > 0:
            global_opt_idx = int(df.loc[r, "opt_idx"])
            x_sol[global_opt_idx] = v

    K_total = milp_build_K_total(y_sol, template_K, profile_order)
    chosen_templates = milp_expand_template_list(y_sol, templates)

    alloc = build_allocation_from_x(
        feasible_option_df=feasible_option_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)

    used_profile_types = sum(1 for p in profile_order if K_total[p] > 0)

    return {
        "method": "MILP-Gurobi(batch)",
        "feasible": True,
        "status": status_str,
        "elapsed": elapsed,
        "gpu_count": int(round(total_gpu.X)),
        "objective": int(round(total_gpu.X)),
        "chosen_templates": chosen_templates,
        "K_total": K_total,
        "x_sol": x_sol,
        "y_sol": y_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": float(total_slack.X),
        "total_remaining_slots": int(round(total_remaining_slots.X)),
        "total_instances": int(round(total_instances.X)),
        "used_profile_types": used_profile_types,
    }


def print_milp_gurobi_batch_result(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nSolver status      : {res['status']}")
    print(f"Total slack        : {res['total_slack']:.6f}")
    print(f"Total instances    : {res['total_instances']}")
    print(f"Total remaining    : {res['total_remaining_slots']}")
    print(f"Used profile types : {res['used_profile_types']}")

    compute_slo_violation_rate(res["alloc"])


def print_milp_template_counts_unified(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])

    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))

# MILP result

In [ ]:
milp_res = solve_milp_gurobi_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    time_limit_s=None,
    mip_gap=None,
    threads=None,
    verbose=False,
)

print_milp_gurobi_batch_result(milp_res)

if milp_res["feasible"]:
    print_milp_template_counts_unified(milp_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, milp_res["x_sol"])

Method        : MILP-Gurobi(batch)
Elapsed (s)   : 43.5957
Feasible      : True
GPU count     : 19
Objective     : 19.0000
Templates     : ['4+1+1+1', '4+1+1+1', '4+1+1+1', '4+1+1+1', '3+3+1', '3+3+1', '2+2+2+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
K_total       : {'7g': 0, '4g': 4, '3g': 4, '2g': 3, '1g': 99}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |   Slack | Placement (batch,profile,count)                  |
|------------|-----------|------------|--------------------|---------|--------------------------------------------------|
| llama      |         3 |     3.0139 |              1.005 |  0.0139 | (1, 4g, 2); (1, 3g, 2)                           |
| gpt2       |       100 |   100.665  |              1.007 |  0.6647 | (1, 1g, 89)                               

# Greedy

In [ ]:
# =========================================================
# Greedy / batch / nofixbs
# Goal hierarchy (heuristic):
#   1) min #GPU   [done by adding one template each round]
#   2) min total_slack
#   3) max total_remaining_slots
#   4) min total_instances
#
# Use PREPARE outputs directly:
#   feasible_option_df, arrival_rate, TEMPLATES, TEMPLATE_K,
#   PROFILE_ORDER, N_WORKLOADS
# =========================================================

import time
import math
import numpy as np
from tabulate import tabulate


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def greedy_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def greedy_build_K_total(y_sol, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            K_total[p] += int(cnt) * int(template_K[t_idx][p])
    return K_total


def print_greedy_template_counts(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])

    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0

    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal

        rows.append([
            o,
            row["workload"],
            int(row["batch"]),
            row["profile"],
            xv,
            f"{mu_o:.6f}",
            f"{subtotal:.6f}",
        ])

    print("\nChosen instances:")
    print(tabulate(
        rows,
        headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"],
        tablefmt="github"
    ))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


# ---------------------------------------------------------
# internal greedy allocation under fixed capacity K_total
# ---------------------------------------------------------
def greedy_allocate_given_capacity(
    feasible_option_df,
    arrival_rate,
    K_total,
    profile_order,
    n_workloads,
    eps=1e-12,
):
    """
    Given total profile capacities K_total, greedily allocate instances.
    Return unified info:
      feasible, x_sol, alloc, provided_rate, total_slack,
      total_instances, total_remaining_slots, used_slots_by_profile
    """

    df = feasible_option_df.copy().reset_index(drop=True)

    # map local row -> global opt_idx
    local_to_global = {r: int(df.loc[r, "opt_idx"]) for r in range(len(df))}

    # residual demand per workload
    residual = np.array(arrival_rate, dtype=float).copy()

    # remaining profile slots
    remain_slots = {p: int(K_total[p]) for p in profile_order}

    # local x first, convert to global opt_idx at end
    x_local = {r: 0 for r in range(len(df))}
    provided = np.zeros(n_workloads, dtype=float)

    # pre-group rows by workload / profile
    rows_by_workload = {i: [] for i in range(n_workloads)}
    rows_by_profile = {p: [] for p in profile_order}

    for r in range(len(df)):
        i = int(df.loc[r, "w_idx"])
        p = df.loc[r, "profile"]
        rows_by_workload[i].append(r)
        rows_by_profile[p].append(r)

    # main greedy loop
    while True:
        unmet = [i for i in range(n_workloads) if residual[i] > eps]
        if len(unmet) == 0:
            break

        best_r = None
        best_key = None

        for r in range(len(df)):
            i = int(df.loc[r, "w_idx"])
            p = df.loc[r, "profile"]
            mu = float(df.loc[r, "mu"])

            if residual[i] <= eps:
                continue
            if remain_slots[p] <= 0:
                continue

            cover = min(mu, residual[i])
            overshoot = max(0.0, mu - residual[i])

            # Greedy ranking:
            # 1) smaller overshoot
            # 2) larger cover
            # 3) larger mu
            # 4) smaller batch (slightly conservative)
            # 5) opt_idx stable tie-break
            key = (
                overshoot,
                -cover,
                -mu,
                int(df.loc[r, "batch"]),
                int(df.loc[r, "opt_idx"]),
            )

            if (best_key is None) or (key < best_key):
                best_key = key
                best_r = r

        # no more usable option but still unmet -> infeasible under this capacity
        if best_r is None:
            break

        i = int(df.loc[best_r, "w_idx"])
        p = df.loc[best_r, "profile"]
        mu = float(df.loc[best_r, "mu"])

        x_local[best_r] += 1
        remain_slots[p] -= 1
        provided[i] += mu
        residual[i] = max(0.0, arrival_rate[i] - provided[i])

    # convert to global opt_idx x_sol
    x_sol = {}
    for r, cnt in x_local.items():
        if cnt > 0:
            x_sol[local_to_global[r]] = int(cnt)

    alloc = build_allocation_from_x(
        feasible_option_df=feasible_option_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)

    feasible = all(w["provided"] + 1e-9 >= w["arrival"] for w in alloc)

    used_slots_by_profile = {}
    for p in profile_order:
        used_slots_by_profile[p] = int(K_total[p] - remain_slots[p])

    total_remaining_slots = int(sum(remain_slots[p] for p in profile_order))
    total_instances = int(sum(x_sol.values()))
    total_slack = float(sum(w["slack"] for w in alloc))
    total_deficit = float(sum(max(0.0, w["arrival"] - w["provided"]) for w in alloc))

    return {
        "feasible": feasible,
        "x_sol": x_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": total_slack,
        "total_deficit": total_deficit,
        "total_instances": total_instances,
        "total_remaining_slots": total_remaining_slots,
        "used_slots_by_profile": used_slots_by_profile,
        "remain_slots_by_profile": remain_slots,
    }


# ---------------------------------------------------------
# try adding exactly one template, choose best candidate
# ---------------------------------------------------------
def greedy_choose_best_next_template(
    feasible_option_df,
    arrival_rate,
    current_y_sol,
    templates,
    template_K,
    profile_order,
    n_workloads,
):
    best = None

    for t_idx in range(len(templates)):
        trial_y = dict(current_y_sol)
        trial_y[t_idx] = trial_y.get(t_idx, 0) + 1

        trial_K_total = greedy_build_K_total(trial_y, template_K, profile_order)

        alloc_res = greedy_allocate_given_capacity(
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            K_total=trial_K_total,
            profile_order=profile_order,
            n_workloads=n_workloads,
        )

        # ranking:
        # 1) feasible first
        # 2) smaller total_slack
        # 3) larger total_remaining_slots
        # 4) smaller total_instances
        # 5) smaller t_idx as stable tie-break
        if alloc_res["feasible"]:
            rank_key = (
                0,
                alloc_res["total_slack"],
                -alloc_res["total_remaining_slots"],
                alloc_res["total_instances"],
                t_idx,
            )
        else:
            rank_key = (
                1,
                alloc_res["total_deficit"],   # smaller deficit is better
                alloc_res["total_instances"],
                t_idx,
            )

        candidate = {
            "t_idx": t_idx,
            "y_sol": trial_y,
            "K_total": trial_K_total,
            "alloc_res": alloc_res,
            "rank_key": rank_key,
        }

        if (best is None) or (candidate["rank_key"] < best["rank_key"]):
            best = candidate

    return best


# ---------------------------------------------------------
# outer greedy solver
# ---------------------------------------------------------
def solve_greedy_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    max_gpu_limit=100,
    verbose=False,
):
    start = time.time()

    # quick sanity check: each workload must have at least one feasible option
    df = feasible_option_df.copy()
    for i in range(n_workloads):
        if not (df["w_idx"] == i).any():
            return {
                "method": "Greedy-batch",
                "feasible": False,
                "elapsed": time.time() - start,
                "status": f"NO_OPTION_FOR_WORKLOAD_{i}",
            }

    y_sol = {}  # template counts

    best_final = None

    for step in range(max_gpu_limit):
        best_next = greedy_choose_best_next_template(
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            current_y_sol=y_sol,
            templates=templates,
            template_K=template_K,
            profile_order=profile_order,
            n_workloads=n_workloads,
        )

        if best_next is None:
            return {
                "method": "Greedy-batch",
                "feasible": False,
                "elapsed": time.time() - start,
                "status": "NO_TEMPLATE_CANDIDATE",
            }

        # accept this best template
        y_sol = best_next["y_sol"]
        K_total = best_next["K_total"]
        alloc_res = best_next["alloc_res"]

        if verbose:
            print(
                f"[Greedy] step={step+1}, add template={templates[best_next['t_idx']][0]}, "
                f"slack={alloc_res['total_slack']:.6f}, "
                f"remaining={alloc_res['total_remaining_slots']}, "
                f"instances={alloc_res['total_instances']}, "
                f"feasible={alloc_res['feasible']}"
            )

        if alloc_res["feasible"]:
            best_final = {
                "y_sol": dict(y_sol),
                "K_total": dict(K_total),
                "alloc_res": alloc_res,
            }
            break

    elapsed = time.time() - start

    if best_final is None:
        return {
            "method": "Greedy-batch",
            "feasible": False,
            "elapsed": elapsed,
            "status": "MAX_GPU_LIMIT_REACHED",
        }

    chosen_templates = greedy_expand_template_list(best_final["y_sol"], templates)
    gpu_count = int(sum(best_final["y_sol"].values()))
    used_profile_types = sum(1 for p in profile_order if best_final["K_total"][p] > 0)

    return {
        "method": "Greedy-batch",
        "feasible": True,
        "elapsed": elapsed,
        "status": "OK",
        "gpu_count": gpu_count,
        "objective": gpu_count,   # align with Oracle/MILP primary objective
        "chosen_templates": chosen_templates,
        "K_total": best_final["K_total"],
        "x_sol": best_final["alloc_res"]["x_sol"],
        "y_sol": best_final["y_sol"],
        "alloc": best_final["alloc_res"]["alloc"],
        "provided_rate": best_final["alloc_res"]["provided_rate"],
        "total_slack": best_final["alloc_res"]["total_slack"],
        "total_remaining_slots": best_final["alloc_res"]["total_remaining_slots"],
        "total_instances": best_final["alloc_res"]["total_instances"],
        "used_profile_types": used_profile_types,
    }


# ---------------------------------------------------------
# unified printer
# ---------------------------------------------------------
def print_greedy_batch_result_unified(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nStatus             : {res['status']}")
    print(f"Total slack        : {res['total_slack']:.6f}")
    print(f"Total remaining    : {res['total_remaining_slots']}")
    print(f"Total instances    : {res['total_instances']}")
    print(f"Used profile types : {res['used_profile_types']}")

    compute_slo_violation_rate(res["alloc"])

# Greedy result

In [ ]:
greedy_res = solve_greedy_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    max_gpu_limit=100,
    verbose=True,   # 想看过程就改成 True
)

print_greedy_batch_result_unified(greedy_res)

if greedy_res["feasible"]:
    print_greedy_template_counts(greedy_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, greedy_res["x_sol"])

# Local Search: Simulated Annealing

In [ ]:
# =========================================================
# Local Search / Simulated Annealing / batch / nofixbs
# From scratch: does NOT depend on greedy result
#
# State: template count vector y_sol
# Objective preference (for best solution keeping):
#   feasible first
#   -> min GPU
#   -> min total_slack
#   -> max total_remaining_slots
#   -> min total_instances
# =========================================================

import time
import math
import random
import numpy as np
from tabulate import tabulate


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def ls_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def ls_build_K_total(y_sol, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            K_total[p] += int(cnt) * int(template_K[t_idx][p])
    return K_total


def print_ls_template_counts(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])
    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0

    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal

        rows.append([
            o,
            row["workload"],
            int(row["batch"]),
            row["profile"],
            xv,
            f"{mu_o:.6f}",
            f"{subtotal:.6f}",
        ])

    print("\nChosen instances:")
    print(tabulate(
        rows,
        headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"],
        tablefmt="github"
    ))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


# ---------------------------------------------------------
# allocation under fixed capacity (independent of greedy)
# ---------------------------------------------------------
def ls_allocate_given_capacity(
    feasible_option_df,
    arrival_rate,
    K_total,
    profile_order,
    n_workloads,
    eps=1e-12,
):
    """
    Independent allocator for local search:
    - repeatedly pick the workload with largest relative unmet ratio
    - for that workload, choose one feasible option that best fits residual
    """

    df = feasible_option_df.copy().reset_index(drop=True)

    local_to_global = {r: int(df.loc[r, "opt_idx"]) for r in range(len(df))}

    rows_by_workload = {i: [] for i in range(n_workloads)}
    for r in range(len(df)):
        i = int(df.loc[r, "w_idx"])
        rows_by_workload[i].append(r)

    remain_slots = {p: int(K_total[p]) for p in profile_order}
    residual = np.array(arrival_rate, dtype=float).copy()
    provided = np.zeros(n_workloads, dtype=float)
    x_local = {r: 0 for r in range(len(df))}

    while True:
        unmet = [i for i in range(n_workloads) if residual[i] > eps]
        if len(unmet) == 0:
            break

        # choose workload with largest relative unmet ratio
        # residual / arrival, tie-break by absolute residual
        target_i = max(
            unmet,
            key=lambda i: (
                residual[i] / max(float(arrival_rate[i]), eps),
                residual[i]
            )
        )

        best_r = None
        best_key = None

        for r in rows_by_workload[target_i]:
            p = df.loc[r, "profile"]
            mu = float(df.loc[r, "mu"])
            b = int(df.loc[r, "batch"])

            if remain_slots[p] <= 0:
                continue

            cover = min(mu, residual[target_i])
            overshoot = max(0.0, mu - residual[target_i])

            # rank for chosen workload:
            # 1) smaller overshoot
            # 2) larger cover
            # 3) prefer larger mu
            # 4) prefer smaller batch
            # 5) stable opt_idx
            key = (
                overshoot,
                -cover,
                -mu,
                b,
                int(df.loc[r, "opt_idx"]),
            )

            if (best_key is None) or (key < best_key):
                best_key = key
                best_r = r

        # if target workload cannot be advanced, try any other unmet workload
        if best_r is None:
            global_best_r = None
            global_best_key = None

            for i in unmet:
                for r in rows_by_workload[i]:
                    p = df.loc[r, "profile"]
                    mu = float(df.loc[r, "mu"])
                    b = int(df.loc[r, "batch"])

                    if remain_slots[p] <= 0:
                        continue

                    cover = min(mu, residual[i])
                    overshoot = max(0.0, mu - residual[i])

                    key = (
                        overshoot,
                        -cover,
                        -mu,
                        b,
                        int(df.loc[r, "opt_idx"]),
                    )

                    if (global_best_key is None) or (key < global_best_key):
                        global_best_key = key
                        global_best_r = r

            if global_best_r is None:
                break
            best_r = global_best_r
            target_i = int(df.loc[best_r, "w_idx"])

        p = df.loc[best_r, "profile"]
        mu = float(df.loc[best_r, "mu"])

        x_local[best_r] += 1
        remain_slots[p] -= 1
        provided[target_i] += mu
        residual[target_i] = max(0.0, float(arrival_rate[target_i]) - provided[target_i])

    x_sol = {}
    for r, cnt in x_local.items():
        if cnt > 0:
            x_sol[local_to_global[r]] = int(cnt)

    alloc = build_allocation_from_x(
        feasible_option_df=feasible_option_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)

    feasible = all(w["provided"] + 1e-9 >= w["arrival"] for w in alloc)

    total_slack = float(sum(w["slack"] for w in alloc))
    total_deficit = float(sum(max(0.0, w["arrival"] - w["provided"]) for w in alloc))
    total_instances = int(sum(x_sol.values()))
    total_remaining_slots = int(sum(remain_slots[p] for p in profile_order))
    used_profile_types = sum(1 for p in profile_order if K_total[p] > 0)

    return {
        "feasible": feasible,
        "x_sol": x_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": total_slack,
        "total_deficit": total_deficit,
        "total_instances": total_instances,
        "total_remaining_slots": total_remaining_slots,
        "used_profile_types": used_profile_types,
        "remain_slots_by_profile": dict(remain_slots),
    }


# ---------------------------------------------------------
# evaluate a template state y_sol
# ---------------------------------------------------------
def ls_evaluate_state(
    y_sol,
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
):
    K_total = ls_build_K_total(y_sol, template_K, profile_order)

    alloc_res = ls_allocate_given_capacity(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        K_total=K_total,
        profile_order=profile_order,
        n_workloads=n_workloads,
    )

    gpu_count = int(sum(y_sol.values()))
    chosen_templates = ls_expand_template_list(y_sol, templates)

    return {
        "y_sol": dict(y_sol),
        "gpu_count": gpu_count,
        "K_total": K_total,
        "chosen_templates": chosen_templates,
        **alloc_res,
    }


# ---------------------------------------------------------
# compare solutions using true preference
# ---------------------------------------------------------
def ls_solution_rank_key(sol):
    return (
        0 if sol["feasible"] else 1,
        sol["gpu_count"] if sol["feasible"] else 10**9,
        sol["total_slack"] if sol["feasible"] else sol["total_deficit"],
        -sol["total_remaining_slots"] if sol["feasible"] else 0,
        sol["total_instances"],
    )


def ls_is_better(sol_a, sol_b):
    if sol_b is None:
        return True
    return ls_solution_rank_key(sol_a) < ls_solution_rank_key(sol_b)


# ---------------------------------------------------------
# scalar energy for SA acceptance
# smaller is better
# ---------------------------------------------------------
def ls_energy(sol):
    if sol["feasible"]:
        # among feasible states:
        # prioritize fewer GPUs strongly, then slack, then reward remaining slots
        return (
            1e9 * sol["gpu_count"]
            + 1e5 * sol["total_slack"]
            - 1e2 * sol["total_remaining_slots"]
            + 1.0 * sol["total_instances"]
        )
    else:
        # infeasible: deficit dominates heavily, then gpu_count
        return (
            1e12
            + 1e8 * sol["total_deficit"]
            + 1e6 * sol["gpu_count"]
            + 1.0 * sol["total_instances"]
        )


# ---------------------------------------------------------
# random feasible initialization from scratch
# ---------------------------------------------------------
def ls_random_initial_solution(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    rng,
    max_gpu_limit=100,
):
    y_sol = {}

    for _ in range(max_gpu_limit):
        t_idx = rng.randrange(len(templates))
        y_sol[t_idx] = y_sol.get(t_idx, 0) + 1

        sol = ls_evaluate_state(
            y_sol=y_sol,
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            templates=templates,
            template_K=template_K,
            profile_order=profile_order,
            n_workloads=n_workloads,
        )

        if sol["feasible"]:
            return sol

    return ls_evaluate_state(
        y_sol=y_sol,
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
    )


# ---------------------------------------------------------
# neighbor generation
# ---------------------------------------------------------
def ls_generate_neighbor(current_y, templates, rng, allow_empty=False):
    y = dict(current_y)
    active = [t for t, c in y.items() if c > 0]
    all_t = list(range(len(templates)))

    moves = ["add", "remove", "swap", "replace"]
    move = rng.choice(moves)

    if move == "add" or len(active) == 0:
        t_add = rng.choice(all_t)
        y[t_add] = y.get(t_add, 0) + 1
        return y, ("add", t_add)

    if move == "remove":
        t_rm = rng.choice(active)
        y[t_rm] -= 1
        if y[t_rm] <= 0:
            del y[t_rm]
        if (not allow_empty) and (sum(y.values()) == 0):
            y[t_rm] = 1
        return y, ("remove", t_rm)

    if move == "swap":
        t_rm = rng.choice(active)
        t_add = rng.choice(all_t)
        y[t_rm] -= 1
        if y[t_rm] <= 0:
            del y[t_rm]
        y[t_add] = y.get(t_add, 0) + 1
        if (not allow_empty) and (sum(y.values()) == 0):
            y[t_add] = 1
        return y, ("swap", t_rm, t_add)

    # replace
    t_rm = rng.choice(active)
    candidates = [t for t in all_t if t != t_rm]
    t_add = rng.choice(candidates) if len(candidates) > 0 else t_rm
    y[t_rm] -= 1
    if y[t_rm] <= 0:
        del y[t_rm]
    y[t_add] = y.get(t_add, 0) + 1
    if (not allow_empty) and (sum(y.values()) == 0):
        y[t_add] = 1
    return y, ("replace", t_rm, t_add)


# ---------------------------------------------------------
# SA main solver
# ---------------------------------------------------------
def solve_local_search_sa_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    seed=42,
    init_temp=5000.0,
    cooling=0.995,
    min_temp=1e-3,
    max_iter=5000,
    max_gpu_limit=100,
    verbose=False,
):
    start = time.time()
    rng = random.Random(seed)

    # random feasible init from scratch
    current = ls_random_initial_solution(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        rng=rng,
        max_gpu_limit=max_gpu_limit,
    )

    best = dict(current)
    T = float(init_temp)

    if verbose:
        print(
            f"[LS-SA][init] feasible={current['feasible']} "
            f"gpu={current['gpu_count']} slack={current['total_slack']:.6f} "
            f"remaining={current['total_remaining_slots']} deficit={current['total_deficit']:.6f}"
        )

    for it in range(max_iter):
        neighbor_y, move_info = ls_generate_neighbor(current["y_sol"], templates, rng, allow_empty=False)

        neighbor = ls_evaluate_state(
            y_sol=neighbor_y,
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            templates=templates,
            template_K=template_K,
            profile_order=profile_order,
            n_workloads=n_workloads,
        )

        e_cur = ls_energy(current)
        e_nxt = ls_energy(neighbor)
        delta = e_nxt - e_cur

        accept = False
        if delta <= 0:
            accept = True
        else:
            prob = math.exp(-delta / max(T, 1e-12))
            if rng.random() < prob:
                accept = True

        if accept:
            current = neighbor

        if ls_is_better(neighbor, best):
            best = dict(neighbor)
            if verbose:
                print(
                    f"[LS-SA][best@{it}] move={move_info} "
                    f"feasible={best['feasible']} gpu={best['gpu_count']} "
                    f"slack={best['total_slack']:.6f} "
                    f"remaining={best['total_remaining_slots']} "
                    f"instances={best['total_instances']}"
                )

        T *= cooling
        if T < min_temp:
            break

    elapsed = time.time() - start

    return {
        "method": "LocalSearch-SA(batch)",
        "feasible": best["feasible"],
        "elapsed": elapsed,
        "status": "OK" if best["feasible"] else "BEST_INFEASIBLE",
        "gpu_count": best["gpu_count"],
        "objective": best["gpu_count"] if best["feasible"] else None,
        "chosen_templates": best["chosen_templates"],
        "K_total": best["K_total"],
        "x_sol": best["x_sol"],
        "y_sol": best["y_sol"],
        "alloc": best["alloc"],
        "provided_rate": best["provided_rate"],
        "total_slack": best["total_slack"],
        "total_deficit": best["total_deficit"],
        "total_remaining_slots": best["total_remaining_slots"],
        "total_instances": best["total_instances"],
        "used_profile_types": best["used_profile_types"],
    }


# ---------------------------------------------------------
# unified printer
# ---------------------------------------------------------
def print_local_search_sa_result_unified(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        print(f"Total deficit  : {res.get('total_deficit', None)}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nStatus             : {res['status']}")
    print(f"Total slack        : {res['total_slack']:.6f}")
    print(f"Total remaining    : {res['total_remaining_slots']}")
    print(f"Total instances    : {res['total_instances']}")
    print(f"Used profile types : {res['used_profile_types']}")

    compute_slo_violation_rate(res["alloc"])

# LS result

In [ ]:
ls_res = solve_local_search_sa_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    seed=42,
    init_temp=5000.0,
    cooling=0.995,
    min_temp=1e-3,
    max_iter=5000,
    max_gpu_limit=100,
    verbose=False,   # 想看搜索过程就改 True
)

print_local_search_sa_result_unified(ls_res)

if ls_res["feasible"]:
    print_ls_template_counts(ls_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, ls_res["x_sol"])

# Greedy 2phase

In [ ]:
# =========================================================
# Two-Phase Greedy / batch / nofixbs
# Phase 1: find min feasible GPU count
# Phase 2: fix GPU count, refine template mix
# =========================================================

import time
import numpy as np
from tabulate import tabulate


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def greedy2_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def greedy2_build_K_total(y_sol, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            K_total[p] += int(cnt) * int(template_K[t_idx][p])
    return K_total


def print_greedy2_template_counts(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])
    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0

    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal

        rows.append([
            o,
            row["workload"],
            int(row["batch"]),
            row["profile"],
            xv,
            f"{mu_o:.6f}",
            f"{subtotal:.6f}",
        ])

    print("\nChosen instances:")
    print(tabulate(
        rows,
        headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"],
        tablefmt="github"
    ))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


# ---------------------------------------------------------
# allocation under fixed capacity
# ---------------------------------------------------------
def greedy2_allocate_given_capacity(
    feasible_option_df,
    arrival_rate,
    K_total,
    profile_order,
    n_workloads,
    eps=1e-12,
):
    df = feasible_option_df.copy().reset_index(drop=True)
    local_to_global = {r: int(df.loc[r, "opt_idx"]) for r in range(len(df))}

    residual = np.array(arrival_rate, dtype=float).copy()
    remain_slots = {p: int(K_total[p]) for p in profile_order}

    x_local = {r: 0 for r in range(len(df))}
    provided = np.zeros(n_workloads, dtype=float)

    rows_by_workload = {i: [] for i in range(n_workloads)}
    for r in range(len(df)):
        i = int(df.loc[r, "w_idx"])
        rows_by_workload[i].append(r)

    while True:
        unmet = [i for i in range(n_workloads) if residual[i] > eps]
        if len(unmet) == 0:
            break

        # pick workload with largest relative deficit
        target_i = max(
            unmet,
            key=lambda i: (residual[i] / max(arrival_rate[i], eps), residual[i])
        )

        best_r = None
        best_key = None

        for r in rows_by_workload[target_i]:
            p = df.loc[r, "profile"]
            mu = float(df.loc[r, "mu"])
            b = int(df.loc[r, "batch"])

            if remain_slots[p] <= 0:
                continue

            cover = min(mu, residual[target_i])
            overshoot = max(0.0, mu - residual[target_i])

            key = (
                overshoot,                  # smaller better
                -cover,                    # larger better
                -mu,                       # larger better
                b,                         # smaller batch slightly preferred
                int(df.loc[r, "opt_idx"]), # stable
            )

            if (best_key is None) or (key < best_key):
                best_key = key
                best_r = r

        # fallback: if target_i cannot move, try any unmet workload
        if best_r is None:
            global_best_r = None
            global_best_key = None

            for i in unmet:
                for r in rows_by_workload[i]:
                    p = df.loc[r, "profile"]
                    mu = float(df.loc[r, "mu"])
                    b = int(df.loc[r, "batch"])

                    if remain_slots[p] <= 0:
                        continue

                    cover = min(mu, residual[i])
                    overshoot = max(0.0, mu - residual[i])

                    key = (
                        overshoot,
                        -cover,
                        -mu,
                        b,
                        int(df.loc[r, "opt_idx"]),
                    )
                    if (global_best_key is None) or (key < global_best_key):
                        global_best_key = key
                        global_best_r = r

            if global_best_r is None:
                break
            best_r = global_best_r
            target_i = int(df.loc[best_r, "w_idx"])

        p = df.loc[best_r, "profile"]
        mu = float(df.loc[best_r, "mu"])

        x_local[best_r] += 1
        remain_slots[p] -= 1
        provided[target_i] += mu
        residual[target_i] = max(0.0, float(arrival_rate[target_i]) - provided[target_i])

    x_sol = {}
    for r, cnt in x_local.items():
        if cnt > 0:
            x_sol[local_to_global[r]] = int(cnt)

    alloc = build_allocation_from_x(
        feasible_option_df=feasible_option_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)

    feasible = all(w["provided"] + 1e-9 >= w["arrival"] for w in alloc)
    total_slack = float(sum(w["slack"] for w in alloc))
    total_deficit = float(sum(max(0.0, w["arrival"] - w["provided"]) for w in alloc))
    total_instances = int(sum(x_sol.values()))
    total_remaining_slots = int(sum(remain_slots[p] for p in profile_order))
    used_profile_types = sum(1 for p in profile_order if K_total[p] > 0)

    return {
        "feasible": feasible,
        "x_sol": x_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": total_slack,
        "total_deficit": total_deficit,
        "total_instances": total_instances,
        "total_remaining_slots": total_remaining_slots,
        "used_profile_types": used_profile_types,
    }


def greedy2_evaluate_y(
    y_sol,
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
):
    K_total = greedy2_build_K_total(y_sol, template_K, profile_order)
    alloc_res = greedy2_allocate_given_capacity(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        K_total=K_total,
        profile_order=profile_order,
        n_workloads=n_workloads,
    )
    return {
        "y_sol": dict(y_sol),
        "gpu_count": int(sum(y_sol.values())),
        "K_total": K_total,
        "chosen_templates": greedy2_expand_template_list(y_sol, templates),
        **alloc_res,
    }


# ---------------------------------------------------------
# ranking
# ---------------------------------------------------------
def greedy2_rank_phase1(sol):
    # phase1: first feasible, otherwise smaller deficit
    return (
        0 if sol["feasible"] else 1,
        sol["gpu_count"] if sol["feasible"] else 10**9,
        sol["total_deficit"],
        sol["total_instances"],
    )


def greedy2_rank_phase2(sol):
    # phase2: fixed GPU count
    return (
        sol["total_slack"],
        -sol["total_remaining_slots"],
        sol["total_instances"],
    )


# ---------------------------------------------------------
# phase 1
# ---------------------------------------------------------
def greedy2_phase1_find_min_gpu(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    max_gpu_limit=100,
    verbose=False,
):
    y_sol = {}

    for step in range(max_gpu_limit):
        best_candidate = None

        for t_idx in range(len(templates)):
            trial_y = dict(y_sol)
            trial_y[t_idx] = trial_y.get(t_idx, 0) + 1

            sol = greedy2_evaluate_y(
                y_sol=trial_y,
                feasible_option_df=feasible_option_df,
                arrival_rate=arrival_rate,
                templates=templates,
                template_K=template_K,
                profile_order=profile_order,
                n_workloads=n_workloads,
            )

            key = greedy2_rank_phase1(sol)
            if (best_candidate is None) or (key < best_candidate["key"]):
                best_candidate = {"sol": sol, "key": key, "t_idx": t_idx}

        if best_candidate is None:
            return None

        y_sol = dict(best_candidate["sol"]["y_sol"])

        if verbose:
            print(
                f"[Greedy2-P1] step={step+1}, add={templates[best_candidate['t_idx']][0]}, "
                f"feasible={best_candidate['sol']['feasible']}, "
                f"deficit={best_candidate['sol']['total_deficit']:.6f}, "
                f"gpu={best_candidate['sol']['gpu_count']}"
            )

        if best_candidate["sol"]["feasible"]:
            return best_candidate["sol"]

    return None


# ---------------------------------------------------------
# phase 2
# ---------------------------------------------------------
def greedy2_phase2_refine_fixed_gpu(
    init_sol,
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    verbose=False,
):
    current = init_sol
    improved = True

    while improved:
        improved = False
        active = [t for t, c in current["y_sol"].items() if c > 0]
        best_neighbor = None

        for t_rm in active:
            for t_add in range(len(templates)):
                if t_add == t_rm:
                    continue

                trial_y = dict(current["y_sol"])
                trial_y[t_rm] -= 1
                if trial_y[t_rm] <= 0:
                    del trial_y[t_rm]
                trial_y[t_add] = trial_y.get(t_add, 0) + 1

                if sum(trial_y.values()) != current["gpu_count"]:
                    continue

                sol = greedy2_evaluate_y(
                    y_sol=trial_y,
                    feasible_option_df=feasible_option_df,
                    arrival_rate=arrival_rate,
                    templates=templates,
                    template_K=template_K,
                    profile_order=profile_order,
                    n_workloads=n_workloads,
                )

                if not sol["feasible"]:
                    continue

                key = greedy2_rank_phase2(sol)
                cur_key = greedy2_rank_phase2(current)

                if key < cur_key:
                    if (best_neighbor is None) or (key < best_neighbor["key"]):
                        best_neighbor = {
                            "sol": sol,
                            "key": key,
                            "move": (t_rm, t_add)
                        }

        if best_neighbor is not None:
            current = best_neighbor["sol"]
            improved = True
            if verbose:
                t_rm, t_add = best_neighbor["move"]
                print(
                    f"[Greedy2-P2] replace {templates[t_rm][0]} -> {templates[t_add][0]}, "
                    f"slack={current['total_slack']:.6f}, "
                    f"remaining={current['total_remaining_slots']}, "
                    f"instances={current['total_instances']}"
                )

    return current


# ---------------------------------------------------------
# main
# ---------------------------------------------------------
def solve_greedy_two_phase_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    max_gpu_limit=100,
    verbose=False,
):
    start = time.time()

    df = feasible_option_df.copy()
    for i in range(n_workloads):
        if not (df["w_idx"] == i).any():
            return {
                "method": "Greedy-2Phase(batch)",
                "feasible": False,
                "elapsed": time.time() - start,
                "status": f"NO_OPTION_FOR_WORKLOAD_{i}",
            }

    phase1_sol = greedy2_phase1_find_min_gpu(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        max_gpu_limit=max_gpu_limit,
        verbose=verbose,
    )

    if phase1_sol is None or (not phase1_sol["feasible"]):
        return {
            "method": "Greedy-2Phase(batch)",
            "feasible": False,
            "elapsed": time.time() - start,
            "status": "PHASE1_FAILED",
        }

    phase2_sol = greedy2_phase2_refine_fixed_gpu(
        init_sol=phase1_sol,
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        verbose=verbose,
    )

    elapsed = time.time() - start

    return {
        "method": "Greedy-2Phase(batch)",
        "feasible": True,
        "elapsed": elapsed,
        "status": "OK",
        "gpu_count": phase2_sol["gpu_count"],
        "objective": phase2_sol["gpu_count"],
        "chosen_templates": phase2_sol["chosen_templates"],
        "K_total": phase2_sol["K_total"],
        "x_sol": phase2_sol["x_sol"],
        "y_sol": phase2_sol["y_sol"],
        "alloc": phase2_sol["alloc"],
        "provided_rate": phase2_sol["provided_rate"],
        "total_slack": phase2_sol["total_slack"],
        "total_deficit": phase2_sol["total_deficit"],
        "total_remaining_slots": phase2_sol["total_remaining_slots"],
        "total_instances": phase2_sol["total_instances"],
        "used_profile_types": phase2_sol["used_profile_types"],
    }


def print_greedy_two_phase_result_unified(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nStatus             : {res['status']}")
    print(f"Total slack        : {res['total_slack']:.6f}")
    print(f"Total remaining    : {res['total_remaining_slots']}")
    print(f"Total instances    : {res['total_instances']}")
    print(f"Used profile types : {res['used_profile_types']}")

    compute_slo_violation_rate(res["alloc"])

# Greedy result

In [ ]:
greedy2_res = solve_greedy_two_phase_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    max_gpu_limit=100,
    verbose=False,
)

print_greedy_two_phase_result_unified(greedy2_res)

if greedy2_res["feasible"]:
    print_greedy2_template_counts(greedy2_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, greedy2_res["x_sol"])

Method        : Greedy-2Phase(batch)
Elapsed (s)   : 68.2901
Feasible      : True
GPU count     : 25
Objective     : 25.0000
Templates     : ['7', '7', '7', '4+1+1+1', '4+1+1+1', '4+1+1+1', '4+1+1+1', '4+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
K_total       : {'7g': 3, '4g': 5, '3g': 0, '2g': 0, '1g': 134}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)                  |
|------------|-----------|------------|--------------------|----------|--------------------------------------------------|
| llama      |         3 |      3.064 |              1.021 |   0.064  | (1, 4g, 4)                                       |
| gpt2       |       100 

# LS 2phase

In [ ]:
# =========================================================
# Two-Phase Simulated Annealing / batch / nofixbs
# Phase 1: find min feasible GPU count
# Phase 2: fix GPU count, optimize secondary objectives
# =========================================================

import time
import math
import random
import numpy as np
from tabulate import tabulate


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def sa2_expand_template_list(y_sol, templates):
    out = []
    for t_idx, cnt in sorted(y_sol.items()):
        out.extend([templates[t_idx][0]] * int(cnt))
    return out


def sa2_build_K_total(y_sol, template_K, profile_order):
    K_total = {p: 0 for p in profile_order}
    for t_idx, cnt in y_sol.items():
        for p in profile_order:
            K_total[p] += int(cnt) * int(template_K[t_idx][p])
    return K_total


def print_sa2_template_counts(y_sol, templates):
    rows = []
    for t_idx, cnt in sorted(y_sol.items()):
        rows.append([t_idx, templates[t_idx][0], cnt])
    print("\nSelected template counts:")
    print(tabulate(rows, headers=["t_idx", "template", "count"], tablefmt="github"))


def print_batch_instance_details(feasible_option_df, x_sol):
    rows = []
    total_mu = 0.0

    chosen = feasible_option_df[feasible_option_df["opt_idx"].isin(x_sol.keys())].copy()
    chosen = chosen.sort_values(["w_idx", "batch", "p_idx"]).reset_index(drop=True)

    for _, row in chosen.iterrows():
        o = int(row["opt_idx"])
        xv = int(x_sol[o])
        mu_o = float(row["mu"])
        subtotal = xv * mu_o
        total_mu += subtotal

        rows.append([
            o,
            row["workload"],
            int(row["batch"]),
            row["profile"],
            xv,
            f"{mu_o:.6f}",
            f"{subtotal:.6f}",
        ])

    print("\nChosen instances:")
    print(tabulate(
        rows,
        headers=["opt_idx", "workload", "batch", "profile", "count", "mu", "count*mu"],
        tablefmt="github"
    ))
    print(f"\nTotal provided throughput = {total_mu:.6f}")


# ---------------------------------------------------------
# allocation under fixed capacity
# ---------------------------------------------------------
def sa2_allocate_given_capacity(
    feasible_option_df,
    arrival_rate,
    K_total,
    profile_order,
    n_workloads,
    eps=1e-12,
):
    df = feasible_option_df.copy().reset_index(drop=True)
    local_to_global = {r: int(df.loc[r, "opt_idx"]) for r in range(len(df))}

    rows_by_workload = {i: [] for i in range(n_workloads)}
    for r in range(len(df)):
        i = int(df.loc[r, "w_idx"])
        rows_by_workload[i].append(r)

    remain_slots = {p: int(K_total[p]) for p in profile_order}
    residual = np.array(arrival_rate, dtype=float).copy()
    provided = np.zeros(n_workloads, dtype=float)
    x_local = {r: 0 for r in range(len(df))}

    while True:
        unmet = [i for i in range(n_workloads) if residual[i] > eps]
        if len(unmet) == 0:
            break

        target_i = max(
            unmet,
            key=lambda i: (
                residual[i] / max(float(arrival_rate[i]), eps),
                residual[i]
            )
        )

        best_r = None
        best_key = None

        for r in rows_by_workload[target_i]:
            p = df.loc[r, "profile"]
            mu = float(df.loc[r, "mu"])
            b = int(df.loc[r, "batch"])

            if remain_slots[p] <= 0:
                continue

            cover = min(mu, residual[target_i])
            overshoot = max(0.0, mu - residual[target_i])

            key = (
                overshoot,
                -cover,
                -mu,
                b,
                int(df.loc[r, "opt_idx"]),
            )

            if (best_key is None) or (key < best_key):
                best_key = key
                best_r = r

        if best_r is None:
            global_best_r = None
            global_best_key = None

            for i in unmet:
                for r in rows_by_workload[i]:
                    p = df.loc[r, "profile"]
                    mu = float(df.loc[r, "mu"])
                    b = int(df.loc[r, "batch"])

                    if remain_slots[p] <= 0:
                        continue

                    cover = min(mu, residual[i])
                    overshoot = max(0.0, mu - residual[i])

                    key = (
                        overshoot,
                        -cover,
                        -mu,
                        b,
                        int(df.loc[r, "opt_idx"]),
                    )

                    if (global_best_key is None) or (key < global_best_key):
                        global_best_key = key
                        global_best_r = r

            if global_best_r is None:
                break

            best_r = global_best_r
            target_i = int(df.loc[best_r, "w_idx"])

        p = df.loc[best_r, "profile"]
        mu = float(df.loc[best_r, "mu"])

        x_local[best_r] += 1
        remain_slots[p] -= 1
        provided[target_i] += mu
        residual[target_i] = max(0.0, float(arrival_rate[target_i]) - provided[target_i])

    x_sol = {}
    for r, cnt in x_local.items():
        if cnt > 0:
            x_sol[local_to_global[r]] = int(cnt)

    alloc = build_allocation_from_x(
        feasible_option_df=feasible_option_df,
        x_sol=x_sol,
        arrival_rate=arrival_rate
    )
    provided_rate = np.array([w["provided"] for w in alloc], dtype=float)

    feasible = all(w["provided"] + 1e-9 >= w["arrival"] for w in alloc)
    total_slack = float(sum(w["slack"] for w in alloc))
    total_deficit = float(sum(max(0.0, w["arrival"] - w["provided"]) for w in alloc))
    total_instances = int(sum(x_sol.values()))
    total_remaining_slots = int(sum(remain_slots[p] for p in profile_order))
    used_profile_types = sum(1 for p in profile_order if K_total[p] > 0)

    return {
        "feasible": feasible,
        "x_sol": x_sol,
        "alloc": alloc,
        "provided_rate": provided_rate,
        "total_slack": total_slack,
        "total_deficit": total_deficit,
        "total_instances": total_instances,
        "total_remaining_slots": total_remaining_slots,
        "used_profile_types": used_profile_types,
    }


def sa2_evaluate_y(
    y_sol,
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    cache=None,
):
    key = tuple(sorted((k, v) for k, v in y_sol.items() if v > 0))
    if cache is not None and key in cache:
        return cache[key]

    K_total = sa2_build_K_total(y_sol, template_K, profile_order)
    alloc_res = sa2_allocate_given_capacity(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        K_total=K_total,
        profile_order=profile_order,
        n_workloads=n_workloads,
    )

    sol = {
        "y_sol": dict(y_sol),
        "gpu_count": int(sum(y_sol.values())),
        "K_total": K_total,
        "chosen_templates": sa2_expand_template_list(y_sol, templates),
        **alloc_res,
    }

    if cache is not None:
        cache[key] = sol
    return sol


# ---------------------------------------------------------
# ranking / energy
# ---------------------------------------------------------
def sa2_rank_phase1(sol):
    return (
        0 if sol["feasible"] else 1,
        sol["gpu_count"] if sol["feasible"] else 10**9,
        sol["total_deficit"],
        sol["total_instances"],
    )


def sa2_rank_phase2(sol):
    return (
        sol["total_slack"],
        -sol["total_remaining_slots"],
        sol["total_instances"],
    )


def sa2_energy_phase1(sol):
    if sol["feasible"]:
        return 1e6 * sol["gpu_count"]
    return 1e12 + 1e8 * sol["total_deficit"] + 1e2 * sol["total_instances"]


def sa2_energy_phase2(sol):
    # fixed GPU
    return (
        1e5 * sol["total_slack"]
        - 1e2 * sol["total_remaining_slots"]
        + 1.0 * sol["total_instances"]
    )


# ---------------------------------------------------------
# initial solution from scratch
# ---------------------------------------------------------
def sa2_random_initial_solution(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    rng,
    max_gpu_limit=100,
    cache=None,
):
    y_sol = {}
    for _ in range(max_gpu_limit):
        t_idx = rng.randrange(len(templates))
        y_sol[t_idx] = y_sol.get(t_idx, 0) + 1

        sol = sa2_evaluate_y(
            y_sol=y_sol,
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            templates=templates,
            template_K=template_K,
            profile_order=profile_order,
            n_workloads=n_workloads,
            cache=cache,
        )
        if sol["feasible"]:
            return sol

    return sa2_evaluate_y(
        y_sol=y_sol,
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        cache=cache,
    )


# ---------------------------------------------------------
# neighbor generation
# ---------------------------------------------------------
def sa2_generate_neighbor_phase1(current_y, templates, rng):
    y = dict(current_y)
    active = [t for t, c in y.items() if c > 0]
    all_t = list(range(len(templates)))

    move = rng.choice(["add", "remove", "swap", "replace"])

    if move == "add" or len(active) == 0:
        t_add = rng.choice(all_t)
        y[t_add] = y.get(t_add, 0) + 1
        return y

    if move == "remove":
        t_rm = rng.choice(active)
        y[t_rm] -= 1
        if y[t_rm] <= 0:
            del y[t_rm]
        if sum(y.values()) == 0:
            y[t_rm] = 1
        return y

    if move == "swap":
        t_rm = rng.choice(active)
        t_add = rng.choice(all_t)
        y[t_rm] -= 1
        if y[t_rm] <= 0:
            del y[t_rm]
        y[t_add] = y.get(t_add, 0) + 1
        return y

    t_rm = rng.choice(active)
    cand = [t for t in all_t if t != t_rm]
    t_add = rng.choice(cand) if len(cand) > 0 else t_rm
    y[t_rm] -= 1
    if y[t_rm] <= 0:
        del y[t_rm]
    y[t_add] = y.get(t_add, 0) + 1
    return y


def sa2_generate_neighbor_phase2(current_y, templates, rng, fixed_gpu):
    y = dict(current_y)
    active = [t for t, c in y.items() if c > 0]
    all_t = list(range(len(templates)))

    if len(active) == 0:
        return y

    t_rm = rng.choice(active)
    t_add = rng.choice(all_t)

    y[t_rm] -= 1
    if y[t_rm] <= 0:
        del y[t_rm]
    y[t_add] = y.get(t_add, 0) + 1

    if sum(y.values()) != fixed_gpu:
        return dict(current_y)
    return y


# ---------------------------------------------------------
# phase 1
# ---------------------------------------------------------
def sa2_phase1_find_min_gpu(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    seed=42,
    init_temp=3000.0,
    cooling=0.995,
    min_temp=1e-3,
    max_iter=3000,
    max_gpu_limit=100,
    verbose=False,
):
    rng = random.Random(seed)
    cache = {}

    current = sa2_random_initial_solution(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        rng=rng,
        max_gpu_limit=max_gpu_limit,
        cache=cache,
    )
    best = dict(current)

    Tcur = float(init_temp)

    for it in range(max_iter):
        neighbor_y = sa2_generate_neighbor_phase1(current["y_sol"], templates, rng)

        if sum(neighbor_y.values()) > max_gpu_limit:
            Tcur *= cooling
            if Tcur < min_temp:
                break
            continue

        neighbor = sa2_evaluate_y(
            y_sol=neighbor_y,
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            templates=templates,
            template_K=template_K,
            profile_order=profile_order,
            n_workloads=n_workloads,
            cache=cache,
        )

        e_cur = sa2_energy_phase1(current)
        e_nxt = sa2_energy_phase1(neighbor)
        delta = e_nxt - e_cur

        accept = False
        if delta <= 0:
            accept = True
        else:
            prob = math.exp(-delta / max(Tcur, 1e-12))
            if rng.random() < prob:
                accept = True

        if accept:
            current = neighbor

        if sa2_rank_phase1(neighbor) < sa2_rank_phase1(best):
            best = dict(neighbor)
            if verbose:
                print(
                    f"[SA2-P1][best@{it}] feasible={best['feasible']} "
                    f"gpu={best['gpu_count']} deficit={best['total_deficit']:.6f}"
                )

        Tcur *= cooling
        if Tcur < min_temp:
            break

    return best


# ---------------------------------------------------------
# phase 2
# ---------------------------------------------------------
def sa2_phase2_refine_fixed_gpu(
    init_sol,
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    fixed_gpu,
    seed=43,
    init_temp=1000.0,
    cooling=0.995,
    min_temp=1e-3,
    max_iter=3000,
    verbose=False,
):
    rng = random.Random(seed)
    cache = {}

    current = dict(init_sol)
    best = dict(init_sol)
    Tcur = float(init_temp)

    for it in range(max_iter):
        neighbor_y = sa2_generate_neighbor_phase2(current["y_sol"], templates, rng, fixed_gpu)

        if sum(neighbor_y.values()) != fixed_gpu:
            Tcur *= cooling
            if Tcur < min_temp:
                break
            continue

        neighbor = sa2_evaluate_y(
            y_sol=neighbor_y,
            feasible_option_df=feasible_option_df,
            arrival_rate=arrival_rate,
            templates=templates,
            template_K=template_K,
            profile_order=profile_order,
            n_workloads=n_workloads,
            cache=cache,
        )

        if not neighbor["feasible"]:
            Tcur *= cooling
            if Tcur < min_temp:
                break
            continue

        e_cur = sa2_energy_phase2(current)
        e_nxt = sa2_energy_phase2(neighbor)
        delta = e_nxt - e_cur

        accept = False
        if delta <= 0:
            accept = True
        else:
            prob = math.exp(-delta / max(Tcur, 1e-12))
            if rng.random() < prob:
                accept = True

        if accept:
            current = neighbor

        if sa2_rank_phase2(neighbor) < sa2_rank_phase2(best):
            best = dict(neighbor)
            if verbose:
                print(
                    f"[SA2-P2][best@{it}] slack={best['total_slack']:.6f} "
                    f"remaining={best['total_remaining_slots']} "
                    f"instances={best['total_instances']}"
                )

        Tcur *= cooling
        if Tcur < min_temp:
            break

    return best


# ---------------------------------------------------------
# main
# ---------------------------------------------------------
def solve_sa_two_phase_batch_unified(
    feasible_option_df,
    arrival_rate,
    templates,
    template_K,
    profile_order,
    n_workloads,
    seed=42,
    phase1_init_temp=3000.0,
    phase1_cooling=0.995,
    phase1_min_temp=1e-3,
    phase1_max_iter=3000,
    phase2_init_temp=1000.0,
    phase2_cooling=0.995,
    phase2_min_temp=1e-3,
    phase2_max_iter=3000,
    max_gpu_limit=100,
    verbose=False,
):
    start = time.time()

    phase1_sol = sa2_phase1_find_min_gpu(
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        seed=seed,
        init_temp=phase1_init_temp,
        cooling=phase1_cooling,
        min_temp=phase1_min_temp,
        max_iter=phase1_max_iter,
        max_gpu_limit=max_gpu_limit,
        verbose=verbose,
    )

    if not phase1_sol["feasible"]:
        return {
            "method": "SA-2Phase(batch)",
            "feasible": False,
            "elapsed": time.time() - start,
            "status": "PHASE1_FAILED",
            "total_deficit": phase1_sol["total_deficit"],
        }

    G_star = phase1_sol["gpu_count"]

    phase2_sol = sa2_phase2_refine_fixed_gpu(
        init_sol=phase1_sol,
        feasible_option_df=feasible_option_df,
        arrival_rate=arrival_rate,
        templates=templates,
        template_K=template_K,
        profile_order=profile_order,
        n_workloads=n_workloads,
        fixed_gpu=G_star,
        seed=seed + 1,
        init_temp=phase2_init_temp,
        cooling=phase2_cooling,
        min_temp=phase2_min_temp,
        max_iter=phase2_max_iter,
        verbose=verbose,
    )

    elapsed = time.time() - start

    return {
        "method": "SA-2Phase(batch)",
        "feasible": True,
        "elapsed": elapsed,
        "status": "OK",
        "gpu_count": phase2_sol["gpu_count"],
        "objective": phase2_sol["gpu_count"],
        "chosen_templates": phase2_sol["chosen_templates"],
        "K_total": phase2_sol["K_total"],
        "x_sol": phase2_sol["x_sol"],
        "y_sol": phase2_sol["y_sol"],
        "alloc": phase2_sol["alloc"],
        "provided_rate": phase2_sol["provided_rate"],
        "total_slack": phase2_sol["total_slack"],
        "total_deficit": phase2_sol["total_deficit"],
        "total_remaining_slots": phase2_sol["total_remaining_slots"],
        "total_instances": phase2_sol["total_instances"],
        "used_profile_types": phase2_sol["used_profile_types"],
    }


def print_sa_two_phase_result_unified(res):
    if not res["feasible"]:
        print("=" * 90)
        print(f"Method         : {res['method']}")
        print(f"Elapsed (s)    : {res['elapsed']:.4f}")
        print("Feasible       : False")
        print(f"Status         : {res.get('status', '-')}")
        print(f"Total deficit  : {res.get('total_deficit', None)}")
        print("=" * 90)
        return

    print_solution_summary(
        method_name=res["method"],
        elapsed=res["elapsed"],
        feasible=res["feasible"],
        objective=res["objective"],
        K_total=res["K_total"],
        templates_used=res["chosen_templates"],
        alloc=res["alloc"],
    )

    print(f"\nStatus             : {res['status']}")
    print(f"Total slack        : {res['total_slack']:.6f}")
    print(f"Total remaining    : {res['total_remaining_slots']}")
    print(f"Total instances    : {res['total_instances']}")
    print(f"Used profile types : {res['used_profile_types']}")

    compute_slo_violation_rate(res["alloc"])

# LS result

In [ ]:
sa2_res = solve_sa_two_phase_batch_unified(
    feasible_option_df=feasible_option_df,
    arrival_rate=arrival_rate,
    templates=TEMPLATES,
    template_K=TEMPLATE_K,
    profile_order=PROFILE_ORDER,
    n_workloads=N_WORKLOADS,
    seed=42,
    phase1_init_temp=3000.0,
    phase1_cooling=0.995,
    phase1_min_temp=1e-3,
    phase1_max_iter=2000,
    phase2_init_temp=1000.0,
    phase2_cooling=0.995,
    phase2_min_temp=1e-3,
    phase2_max_iter=2000,
    max_gpu_limit=100,
    verbose=False,
)

print_sa_two_phase_result_unified(sa2_res)

if sa2_res["feasible"]:
    print_sa2_template_counts(sa2_res["y_sol"], TEMPLATES)
    print_batch_instance_details(feasible_option_df, sa2_res["x_sol"])

Method        : SA-2Phase(batch)
Elapsed (s)   : 68.3345
Feasible      : True
GPU count     : 20
Objective     : 20.0000
Templates     : ['7', '7', '7', '4+1+1+1', '4+1+1+1', '4+1+1+1', '4+1+1+1', '4+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1', '1+1+1+1+1+1+1']
K_total       : {'7g': 3, '4g': 5, '3g': 0, '2g': 0, '1g': 99}

Per-workload allocation / rate summary:
| Workload   |   Arrival |   Provided |   Provided/Arrival |    Slack | Placement (batch,profile,count)                  |
|------------|-----------|------------|--------------------|----------|--------------------------------------------------|
| llama      |         3 |      3.064 |              1.021 |   0.064  | (1, 4g, 4)                                       |
| gpt2       |       100 |    100.665 |              1.007 |   0.6647 | (1, 1g, 89)                                